# Generalized Diffusion on 2D Ferromagnetic Ising: Correlated Case

This notebook runs the **correlated / interaction-aware version** of the generalized diffusion framework on the **2D ferromagnetic Ising model** on a `50 x 50` lattice.

## What this notebook does

- Loads equilibrium initial states from the FPGA-generated 2D ferromagnet dataset
- Builds the nearest-neighbor 2D ferromagnetic coupling matrix
- Runs the **correlated forward noising process** with:
  - `turn_corr_off = False`
  - `h_ind = 0.0`
- Trains an MLP to predict per-site clean-state probabilities from noisy states
- Runs the **correlated reverse sampler** with:
  - `set_ind_rev_sampler = False`
- Produces reverse samples and comparison plots against the physical reference data

## Expected input files

This notebook expects the FPGA dataset directory:

- `L50_Results_noising_100000MCS_81beta_s46_nobias/`

containing files of the form:

- `Trial_*.mat`

At the current default settings:
- `grid_size = 50`
- `N = 2500`
- `T = 101`
- `beta_start = 0.453125`
- `num_systems = 10000`

the notebook loads **one replica per trial file** at the selected `beta_start`.

## Where outputs are saved

Outputs are saved in the **current working directory**.

Typical saved outputs include:

- model checkpoint: `ckpt_2dfm_*.pth`
- loss curve PDF: `plot_2_losscurve_0p453125_corr.pdf`
- forward / reverse trajectory PDFs
- energy / magnetization / overlap PDFs

## Training vs checkpoint loading

- Set `USE_SAVED_MODEL = False` to run the full pipeline:
  forward diffusion -> training -> reverse sampling
- Set `USE_SAVED_MODEL = True` to skip training and load a saved checkpoint
- Set `CHECKPOINT_PATH = None` to auto-generate or auto-discover checkpoints
- Or set `CHECKPOINT_PATH = "your_checkpoint_file.pth"` to load a specific checkpoint

This notebook is the **main generalized correlated formulation**, where the known Ising couplings are used in both the forward and reverse diffusion kernels.

In [ ]:
import os

# Ensure working directory is the repo root so data paths resolve correctly
os.chdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), "..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())

print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))

import torch
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
print("device name:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# CONFIGURATION: Training vs Inference-Only Mode
# ============================================================
# Set to True to load a saved checkpoint and skip to inference
USE_SAVED_MODEL = False

# Path to checkpoint file (auto-generated if None)
CHECKPOINT_PATH = None

In [ ]:
# ============================================================
# Load Checkpoint for Inference-Only Mode (if enabled)
# ============================================================

if USE_SAVED_MODEL:
    import torch
    import torch.nn as nn
    import numpy as np
    import os
    import glob
    
    if CHECKPOINT_PATH is None:
        print("No checkpoint path specified. Searching for available checkpoints...")
        available_ckpts = glob.glob("ckpt_3dsg_*.pth")
        if len(available_ckpts) == 0:
            raise FileNotFoundError("No checkpoint files found matching 'ckpt_3dsg_*.pth'. "
                                  "Please specify CHECKPOINT_PATH manually or train a new model.")
        print(f"Found {len(available_ckpts)} checkpoint(s):")
        for i, ckpt in enumerate(sorted(available_ckpts)):
            print(f"  [{i}] {ckpt}")
        raise ValueError("Multiple checkpoints found. Please set CHECKPOINT_PATH to one of the above files.")
    
    if not os.path.exists(CHECKPOINT_PATH):
        raise FileNotFoundError(f"Checkpoint file '{CHECKPOINT_PATH}' not found. "
                              f"Please set USE_SAVED_MODEL=False to train a new model, "
                              f"or check that the file path is correct.")
    
    print(f"Loading checkpoint from {CHECKPOINT_PATH}...")
    
    checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    
    N = checkpoint["N"]
    T = checkpoint["T"]
    h_ind = checkpoint["h_ind"]
    turn_corr_off = checkpoint["turn_corr_off"]
    set_ind_rev_sampler = checkpoint["set_ind_rev_sampler"]
    beta_schedule = checkpoint["beta_schedule"]
    beta_start = checkpoint.get("beta_start", 1.0)
    num_systems = checkpoint.get("num_systems", "unknown")
    
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
    print(f"  Best train loss: {checkpoint['best_train_loss']:.5f}")
    print(f"  Best val loss: {checkpoint['best_val_loss']:.5f}")
    print(f"  N={N}, T={T}, beta_start={beta_start}, num_systems={num_systems}")
    print(f"  h_ind={h_ind}, turn_corr_off={turn_corr_off}, set_ind_rev_sampler={set_ind_rev_sampler}")
    print("\nNote: The model will be moved to the appropriate device after device setup.")
    print("      Skip forward diffusion and training cells, go directly to reverse sampling.")
else:
    print("USE_SAVED_MODEL=False: Will run forward diffusion and training as normal.")

In [ ]:
# Correlated forward and reverse diffusion processes with p-bits (N-spin)

import socket
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import h5py

# ---- Device & system info ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print("Node Name:", socket.gethostname())
print("PyTorch Version:", torch.__version__)
print("CUDA Version Used by PyTorch:", torch.version.cuda)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Device Count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"CUDA Device {i}: {torch.cuda.get_device_name(i)}")
print(f"Using device: {device}")
print("=" * 60)

# Reproducibility across NumPy + PyTorch
rng = np.random.default_rng(926)
torch.manual_seed(926)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(926)

def enumerate_states(N: int) -> np.ndarray:
    """All 2^N possible configurations of an N-spin Ising system as bipolar vectors."""
    NUM_STATES = 1 << N
    STATES = np.empty((NUM_STATES, N), dtype=int)
    for idx in range(NUM_STATES):
        for i in range(N):
            bit = (idx >> (N - 1 - i)) & 1
            STATES[idx, i] = 1 if bit == 1 else -1
    return STATES

def state_to_index(s: np.ndarray) -> int:
    """Convert bipolar spin vector to integer index."""
    idx = 0
    N = len(s)
    for i in range(N):
        if s[i] > 0:
            idx |= (1 << (N - 1 - i))
    return idx

def index_to_state(idx: int, N: int) -> np.ndarray:
    """Convert index to bipolar spin vector of length N."""
    s = np.empty(N, dtype=int)
    for i in range(N):
        bit = (idx >> (N - 1 - i)) & 1
        s[i] = 1 if bit == 1 else -1
    return s

def eta_to_beta_schedule(eta_schedule: np.ndarray, eps: float = 1e-9) -> np.ndarray:
    """
    β_t = arctanh(1 - 2 η_t); clip to avoid 0 causing infinities.
    Use with h_ind=1 for independent noising for β parameterization.
    """
    eta = np.asarray(eta_schedule, dtype=float)
    eta = np.clip(eta, eps, 0.5)
    return np.arctanh(1.0 - 2.0 * eta)

def beta_to_eta_schedule(beta_schedule: np.ndarray) -> np.ndarray:
    """
    Inverse of eta_to_beta_schedule:
    eta = (1 - tanh(beta)) / 2
    """
    beta = np.asarray(beta_schedule, dtype=float)
    return 0.5 * (1.0 - np.tanh(beta))

In [ ]:
# Display the 81 FPGA beta values:
# FPGA formula: k_vals = 80:-1:0, beta_i = k / 64

k_vals = np.arange(80, -1, -1)  # [80, 79, 78, ..., 1, 0]
full_beta_schedule = k_vals / 64  # β_i = k/64

print("The 81 FPGA beta values:")
print(f"Formula: β_i = k/64, where k ∈ [80, 79, ..., 1, 0]")
print(f"\nAll 81 values:")
for i, (k, beta) in enumerate(zip(k_vals, full_beta_schedule)):
    print(f"  [{i:2d}] k={k:2d}, β = {beta:.6f} (= {k}/64)")
    
print(f"\nRange: β ∈ [{full_beta_schedule[-1]:.6f}, {full_beta_schedule[0]:.6f}]")
print(f"Step size: {1/64:.6f} (1/64)")

In [ ]:
### Loaders for 3D spin glass states and J

def load_states_txt(path: str, map_to_bipolar=True) -> np.ndarray:
    """
    Each line is a bitstring of length N (e.g., '0101...'), one sample per line.
    Returns shape (num_samples, N), values in {+1, -1} if map_to_bipolar else {0,1}.
    """
    with open(path, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    arr01 = np.array([[1 if c=='1' else 0 for c in ln] for ln in lines], dtype=int)
    if not map_to_bipolar:
        return arr01
    return np.where(arr01==1, 1, -1).astype(int)

def infer_3d_dims_from_N(N: int) -> tuple[int,int,int]:
    """
    Try to infer a cube Lx=Ly=Lz if possible, else fall back to (N,1,1).
    For your file: N=27 -> (3,3,3).
    """
    L = round(N ** (1/3))
    if L**3 == N:
        return L, L, L

    return N, 1, 1

def _idx3d(x, y, z, Lx, Ly, Lz):
    return (z*Ly + y)*Lx + x

def build_J_from_bonds_3d(Lx:int, Ly:int, Lz:int, Jx:np.ndarray, Jy:np.ndarray, Jz:np.ndarray) -> np.ndarray:
    """
    Jx, Jy, Jz are bond couplings to +x, +y, +z neighbors respectively:
      shapes: Jx(Lx-1,Ly,Lz), Jy(Lx,Ly-1,Lz), Jz(Lx,Ly,Lz-1)
    Returns symmetric N×N J with zero diagonal.
    """
    N = Lx*Ly*Lz
    J = np.zeros((N,N), dtype=float)
    # +x bonds
    for z in range(Lz):
        for y in range(Ly):
            for x in range(Lx-1):
                i = _idx3d(x,   y, z, Lx, Ly, Lz)
                j = _idx3d(x+1, y, z, Lx, Ly, Lz)
                w = float(Jx[x,y,z])
                J[i,j] = J[j,i] = w
    # +y bonds
    for z in range(Lz):
        for y in range(Ly-1):
            for x in range(Lx):
                i = _idx3d(x, y,   z, Lx, Ly, Lz)
                j = _idx3d(x, y+1, z, Lx, Ly, Lz)
                w = float(Jy[x,y,z])
                J[i,j] = J[j,i] = w
    # +z bonds
    for z in range(Lz-1):
        for y in range(Ly):
            for x in range(Lx):
                i = _idx3d(x, y, z,   Lx, Ly, Lz)
                j = _idx3d(x, y, z+1, Lx, Ly, Lz)
                w = float(Jz[x,y,z])
                J[i,j] = J[j,i] = w
    return J

def load_J_any(J_path: str|None, N_expected: int, Lx:int, Ly:int, Lz:int) -> np.ndarray:
    """
    Load J from:
      - .npy: N×N float array
      - .npz: either 'J' (N×N) or ('Jx','Jy','Jz') bond arrays
      - .mat: MATLAB file with 'J' variable or ('Jx','Jy','Jz')
      - .txt: lines 'i j w' (0-based indices), symmetric implied
    """
    if J_path is None:
        raise FileNotFoundError("Please set J_PATH to your attached J matrix file.")

    low = J_path.lower()
    if low.endswith(".npy"):
        J = np.load(J_path)
        assert J.shape == (N_expected, N_expected), f"J shape {J.shape} != {(N_expected,N_expected)}"
        np.fill_diagonal(J, 0.0)
        return J.astype(float)

    if low.endswith(".npz"):
        data = np.load(J_path)
        if "J" in data:
            J = data["J"].astype(float)
            assert J.shape == (N_expected, N_expected)
            np.fill_diagonal(J, 0.0)
            return J
        # bonds
        Jx, Jy, Jz = data["Jx"], data["Jy"], data["Jz"]
        return build_J_from_bonds_3d(Lx, Ly, Lz, Jx, Jy, Jz)
    
    if low.endswith(".mat"):
        import scipy.io
        import scipy.sparse

        try:
            data = scipy.io.loadmat(J_path)
        except NotImplementedError:
            data = None

        if data is not None and "J" in data:
            J = data["J"]
            if scipy.sparse.issparse(J):
                J = J.toarray()
            J = J.astype(float)
        else:
            # v7.3 (HDF5) path
            with h5py.File(J_path, "r") as f:
                if "J" not in f:
                    # bond-form fallback
                    if all(k in f for k in ("Jx","Jy","Jz")):
                        Jx = np.array(f["Jx"]); Jy = np.array(f["Jy"]); Jz = np.array(f["Jz"])
                        return build_J_from_bonds_3d(Lx, Ly, Lz, Jx, Jy, Jz)
                    raise ValueError("MAT file must contain 'J' or ('Jx','Jy','Jz').")

                J_item = f["J"]

                # v7.3 sparse format
                if isinstance(J_item, h5py.Group) and all(k in J_item for k in ("data","ir","jc")):
                    data_vals = np.array(J_item["data"]).ravel().astype(float)
                    ir = np.array(J_item["ir"]).ravel().astype(np.int64)   # 0-based row indices
                    jc = np.array(J_item["jc"]).ravel().astype(np.int64)   # column pointers (0..nnz)
                    # shape is len(jc)-1 by len(jc)-1 for a square matrix
                    ncols = len(jc) - 1
                    J_sparse = scipy.sparse.csc_matrix((data_vals, ir, jc), shape=(ncols, ncols))
                    J = J_sparse.toarray()
                else:
                    # dense dataset fallback
                    J = np.array(J_item).astype(float)
                    if J.shape != (N_expected, N_expected) and J.T.shape == (N_expected, N_expected):
                        J = J.T

        # --- normalize to the format we want ---
        if J.shape != (N_expected, N_expected):
            raise ValueError(f"J shape {J.shape} != {(N_expected, N_expected)}")

        # If the file stored only one triangle, mirror it WITHOUT dividing by 2
        upper_nnz = np.count_nonzero(np.triu(J, 1))
        lower_nnz = np.count_nonzero(np.tril(J, 1))
        if upper_nnz > 0 and lower_nnz == 0:
            J = J + J.T
        elif lower_nnz > 0 and upper_nnz == 0:
            J = J + J.T
        # If both halves exist but are not exactly equal, symmetrize
        elif not np.allclose(J, J.T):
            J = 0.5 * (J + J.T)

        # Restore ±1 values if halved to ±0.5
        if np.isclose(np.max(np.abs(J)), 0.5):
            J *= 2.0

        # Zero diagonal, clip numerical fuzz
        np.fill_diagonal(J, 0.0)
        J[np.abs(J) < 1e-12] = 0.0
        return J
 
    # edge list: i j w
    if low.endswith(".txt"):
        J = np.zeros((N_expected, N_expected), dtype=float)
        with open(J_path, "r") as f:
            for ln in f:
                ln = ln.strip()
                if not ln or ln.startswith("#"):
                    continue
                i,j,w = ln.split()
                i,j = int(i), int(j)
                w = float(w)
                J[i,j] = J[j,i] = w
        np.fill_diagonal(J, 0.0)
        return J

    raise ValueError(f"Unsupported J file: {J_path}")


In [ ]:
def build_neighbor_list(J: np.ndarray, device_):
    """
    Build sparse neighbor list from J matrix for efficient local field computation.
    
    For 3D lattice spin glasses, this reduces O(N^2) to O(N*deg) where deg ~ 6.
    
    Args:
        J: (N, N) coupling matrix (NumPy)
        device_: torch device
    
    Returns:
        nbr_idx: (N, max_deg) int64 tensor on device, neighbor indices (-1 for padding)
        nbr_w: (N, max_deg) float32 tensor on device, coupling weights (0 for padding)
    """
    N = J.shape[0]
    J_coo = torch.as_tensor(J, dtype=torch.float32, device='cpu')
    
    nbr_idx = []
    nbr_w = []
    for j in range(N):
        # Find nonzero off-diagonal couplings
        nz = (J_coo[j] != 0).nonzero(as_tuple=False).squeeze(1)
        nz = nz[nz != j]  # exclude diagonal
        nbr_idx.append(nz.to(torch.int64))
        nbr_w.append(J_coo[j, nz])
    
    # Pad to max degree
    deg = max(x.numel() for x in nbr_idx) if nbr_idx else 0
    if deg == 0:
        # J is all zeros or only diagonal
        return None, None
    
    idx_pad = torch.full((N, deg), -1, dtype=torch.int64)
    w_pad = torch.zeros((N, deg), dtype=torch.float32)
    for j in range(N):
        d = nbr_idx[j].numel()
        if d > 0:
            idx_pad[j, :d] = nbr_idx[j]
            w_pad[j, :d] = nbr_w[j]
    
    return idx_pad.to(device_), w_pad.to(device_)


In [ ]:
def build_single_update_matrix(beta: float, which_spin: int, J: np.ndarray, h: np.ndarray, STATES: np.ndarray) -> np.ndarray:
    """
    Return the transition matrix (2^N x 2^N) that updates exactly one spin
    using Gibbs conditional with general J (NxN) and bias h (N,). Columns=current, rows=next.
    """
    N = J.shape[0]
    NUM_STATES = STATES.shape[0]
    W = np.zeros((NUM_STATES, NUM_STATES), dtype=float)
    i = which_spin

    for col in range(NUM_STATES):
        s = STATES[col]
        I_i = float(J[i, :].dot(s) - J[i, i] * s[i] + h[i])  # exclude J[i,i] self term
        p_plus = 0.5 * (1.0 + np.tanh(beta * I_i))
        s_plus, s_minus = s.copy(), s.copy()
        s_plus[i], s_minus[i] = +1, -1
        row_plus = state_to_index(s_plus)
        row_minus = state_to_index(s_minus)
        W[row_plus,  col] = p_plus
        W[row_minus, col] = 1.0 - p_plus
    return W

def build_gibbs_matrix(beta: float, J: np.ndarray, h: np.ndarray, STATES: np.ndarray, order=None) -> np.ndarray:
    """Compose single-spin update matrices in the specified order: W_Gibbs = W_{order[-1]} @ ... @ W_{order[0]}."""
    N = J.shape[0]
    if order is None:
        order = tuple(range(N)) # default sequential order
    NUM_STATES = STATES.shape[0]
    W = np.eye(NUM_STATES)
    for which in order:
        W = build_single_update_matrix(beta, which, J, h, STATES) @ W
    return W

In [ ]:
def calculate_probability_evolution(initial_prob: np.ndarray, beta_schedule: np.ndarray, J: np.ndarray, h: np.ndarray, T: int, STATES: np.ndarray, W_list=None):
    """
    Probability-space evolution π_t = W_{t-1} @ π_{t-1}; returns (means[T,N], pairs[T,N,N], probs[T,2^N]).
    """
    NUM_STATES, N = STATES.shape[0], STATES.shape[1]
    means = np.zeros((T, N))
    pairs = np.zeros((T, N, N))
    probs = np.zeros((T, NUM_STATES))
    p = initial_prob.copy()
    probs[0] = p
    means[0] = (p[:, None] * STATES).sum(axis=0)
    for i in range(N):
        for j in range(N):
            pairs[0, i, j] = (p * STATES[:, i] * STATES[:, j]).sum()
    for t in range(1, T):
        Wt = W_list[t-1] if W_list is not None else build_gibbs_matrix(beta_schedule[t-1], J, h, STATES)
        p = Wt @ p
        probs[t] = p
        means[t] = (p[:, None] * STATES).sum(axis=0)
        for i in range(N):
            for j in range(N):
                pairs[t, i, j] = (p * STATES[:, i] * STATES[:, j]).sum()
    return means, pairs, probs

def forward_diffusion_process(s0: np.ndarray, beta_schedule: np.ndarray, J: np.ndarray, h: np.ndarray, T: int, rng, STATES: np.ndarray, W_list=None):
    """
    Sample a forward trajectory s_0..s_{T-1} using transition matrices (column-stochastic).
    """
    NUM_STATES = STATES.shape[0]
    traj = [None] * T
    traj[0] = s0.copy()
    onehot = np.eye(NUM_STATES)[state_to_index(s0)]
    for t in range(1, T):
        Wt = W_list[t-1] if W_list is not None else build_gibbs_matrix(beta_schedule[t-1], J, h, STATES)
        pi_t = (Wt @ onehot).ravel()
        idx = rng.choice(NUM_STATES, p=pi_t)
        traj[t] = STATES[idx]
        onehot = np.eye(NUM_STATES)[idx]
    return traj

def posterior_prob_correlated_detailed(s_t: np.ndarray, s0: np.ndarray, beta_schedule: np.ndarray, J: np.ndarray, h: np.ndarray, t: int, STATES: np.ndarray, W_list=None):
    """
    Exact posterior for one reverse step:
    P(s_{t-1}|s_t,s_0) ∝ [W_{t-1}[s_t, :]] ⊙ π_{t-1}, where π_{t-1} is forward-evolved from s0.
    Returns (likelihood[2^N], prior[2^N], Z, posterior[2^N]).
    """
    NUM_STATES = STATES.shape[0]
    s0_idx = state_to_index(s0)
    pi = np.eye(NUM_STATES)[s0_idx].astype(float)
    for k in range(t-1):
        Wk = W_list[k] if W_list is not None else build_gibbs_matrix(beta_schedule[k], J, h, STATES)
        pi = Wk @ pi
    Wt = W_list[t-1] if W_list is not None else build_gibbs_matrix(beta_schedule[t-1], J, h, STATES)
    s_t_idx = state_to_index(s_t)
    likelihood = Wt[s_t_idx, :].copy()
    prior = pi.copy()
    Z = float(likelihood @ prior)
    posterior = (likelihood * prior) / Z
    return likelihood, prior, Z, posterior
def avg_trajectory_and_pairs(trajectories, device=None, dtype=torch.float32):
    """
    trajectories: (S, T, N) as np.ndarray or torch.Tensor with spins in {-1, +1}
    Returns:
        means, pairs as torch.Tensors on `device`
    """
    if not isinstance(trajectories, torch.Tensor):
        trajectories = torch.from_numpy(trajectories)

    if device is None:
        device = trajectories.device

    traj = trajectories.to(device=device, dtype=dtype)  # (S, T, N)
    S = traj.shape[0]

    means = traj.mean(dim=0)                            # (T, N)
    pairs = torch.einsum('sti,stj->tij', traj, traj) / S  # (T, N, N)

    return means, pairs

In [ ]:
def sample_boltzmann_initial_states(num_systems: int, J: np.ndarray, h: np.ndarray, beta_start: float, rng, STATES: np.ndarray):
    """Sample s0 from Boltzmann P(s) ∝ exp[-β E(s)], E(s) = -½ sᵀJs - hᵀs. Returns (s0_all[num_systems,N], p0[2^N])."""
    energies = -0.5 * np.einsum('si,ij,sj->s', STATES, J, STATES) - STATES.dot(h)
    w = np.exp(-beta_start * energies)
    p0 = w / w.sum()
    idx = rng.choice(STATES.shape[0], size=num_systems, p=p0)
    return STATES[idx], p0

def pbit_step_both_batch(
    m_prev: torch.Tensor,          # (B, N), ±1
    J: torch.Tensor | None,        # (N, N) or None for J=0
    h: torch.Tensor,               # (N,)
    beta_vec,                      # scalar float or 1D tensor length N
    h_ind=1.0,                     # scalar or 1D tensor length N
    use_s0_for_ind: bool = False,
    s0: torch.Tensor | None = None, # (B, N) if use_s0_for_ind
    nbr_idx: torch.Tensor | None = None,  # (N, max_deg) neighbor indices
    nbr_w: torch.Tensor | None = None     # (N, max_deg) neighbor weights
) -> torch.Tensor:
    """
    Batched p-bit step function for batched spin systems.
    
    Parallelization strategy:
    - Sequential in spin index i (physics requirement - Gibbs sweep)
    - Parallel across batch dimension B (systems/chains)
    
    This maintains identical physics while batching independent systems.
    The device is determined by the input tensors.
    
    Args:
        m_prev: (B, N) tensor of ±1 spins
        J: (N, N) coupling matrix or None for J=0 (ignored if nbr_idx provided)
        h: (N,) bias vector
        beta_vec: inverse temperature (scalar or per-spin)
        h_ind: independent field strength
        use_s0_for_ind: if True, use s0 for independent term; else use m_prev
        s0: (B, N) reference state for independent term
        nbr_idx: (N, max_deg) neighbor indices for sparse computation (-1 for padding)
        nbr_w: (N, max_deg) neighbor weights for sparse computation
    
    Returns:
        (B, N) tensor of updated spins
    """
    B, N = m_prev.shape
    device_ = m_prev.device
    m_cur = m_prev.clone()

    # β_t: scalar or per-spin
    if not torch.is_tensor(beta_vec):
        beta_vec = torch.full((N,), float(beta_vec), device=device_, dtype=torch.float32)
    elif beta_vec.ndim == 0:
        beta_vec = beta_vec.expand(N)
    else:
        beta_vec = beta_vec.to(device_, dtype=torch.float32)

    # h_ind: scalar or per-spin
    if not torch.is_tensor(h_ind):
        h_ind = torch.full((N,), float(h_ind), device=device_, dtype=torch.float32)
    elif h_ind.ndim == 0:
        h_ind = h_ind.expand(N)
    else:
        h_ind = h_ind.to(device_, dtype=torch.float32)

    base = s0 if use_s0_for_ind else m_prev   # (B, N)

    # Use sparse neighbor list if provided (exact same result, much faster for sparse J)
    use_sparse = (nbr_idx is not None and nbr_w is not None)

    for j in range(N):
        # correlated term: (B,)
        if use_sparse:
            # Sparse computation using neighbor list (O(deg) instead of O(N))
            nbrs = nbr_idx[j]  # (max_deg,)
            mask = nbrs >= 0
            if mask.any():
                nbrs_valid = nbrs[mask]
                w = nbr_w[j][mask]
                dep_field = (m_cur[:, nbrs_valid] * w).sum(dim=1) + h[j]
            else:
                dep_field = h[j].expand(B)
        elif J is not None:
            # Dense computation (fallback)
            dep_field = torch.matmul(m_cur, J[j]) + h[j]
        else:
            dep_field = h[j].expand(B)

        ind_field = h_ind[j] * base[:, j]
        local = dep_field + ind_field          # (B,)

        tanh_arg = torch.tanh(beta_vec[j] * local)
        u = 2.0 * torch.rand(B, device=device_) - 1.0
        m_cur[:, j] = torch.where(
            tanh_arg + u >= 0.0,
            torch.ones_like(local),
            -torch.ones_like(local),
        )

    return m_cur

def sample_initial_states_with_pbits(num_systems: int, J: np.ndarray, h: np.ndarray,
                                     beta_start: float, rng, N: int,
                                     init_burn_in: int, int_burn_in: int,
                                     nbr_idx=None, nbr_w=None) -> np.ndarray:
    """
    Use p-bits to draw num_systems samples at (J,h,beta_start).
    Uses unified torch function on the default device.
    """
    # Start on device
    J_t = torch.from_numpy(J.astype(np.float32)).to(device) if nbr_idx is None else None
    h_t = torch.from_numpy(h.astype(np.float32)).to(device)
    
    # Random start
    cur = torch.from_numpy(rng.choice([-1, 1], size=(1, N)).astype(np.float32)).to(device)

    # Initial burn-in
    for _ in range(init_burn_in):
        cur = pbit_step_both_batch(cur, J_t, h_t, beta_start, h_ind=0.0, nbr_idx=nbr_idx, nbr_w=nbr_w)

    # Collect samples with spacing
    out = []
    for k in range(num_systems):
        out.append(cur[0].cpu().numpy().astype(int))
        for _ in range(int_burn_in):
            cur = pbit_step_both_batch(cur, J_t, h_t, beta_start, h_ind=0.0, nbr_idx=nbr_idx, nbr_w=nbr_w)

    return np.array(out)

def pbit_forward_diffusion_all(
    s0_all_t: torch.Tensor,        # (num_systems, N), ±1
    beta_schedule,                 # np.ndarray or torch.tensor length T-1
    J_t: torch.Tensor | None,      # (N, N) or None
    h_t: torch.Tensor,             # (N,)
    T: int,
    h_ind=1.0,
    use_s0_for_ind: bool = False,
    nbr_idx: torch.Tensor | None = None,
    nbr_w: torch.Tensor | None = None,
) -> torch.Tensor:
    """
    Forward diffusion for batched spin systems.
    
    Parallelization strategy:
    - Parallel across all systems (batch dimension)
    - Sequential across timesteps (physics requirement)
    - Sequential across spins within each timestep (Gibbs sweep)
    
    The device is determined by the input tensors.
    
    Args:
        s0_all_t: (num_systems, N) initial states
        beta_schedule: temperature schedule
        J_t: (N, N) coupling matrix or None
        h_t: (N,) bias
        T: number of timesteps
        h_ind: independent field strength (scalar, per-spin vector, or schedule of length T-1)
        use_s0_for_ind: use s0 vs s^{t-1} for independent term
        nbr_idx: (N, max_deg) neighbor indices for sparse computation
        nbr_w: (N, max_deg) neighbor weights for sparse computation
    
    Returns:
        (num_systems, T, N) tensor of trajectories
    """
    device_ = s0_all_t.device

    if not torch.is_tensor(beta_schedule):
        beta_schedule_t = torch.from_numpy(
            np.asarray(beta_schedule, dtype=np.float32)
        ).to(device_)
    else:
        beta_schedule_t = beta_schedule.to(device_, dtype=torch.float32)

    # Handle h_ind schedule
    h_ind_is_schedule = False
    h_ind_t_schedule = None
    
    # Check if h_ind is a sequence with length matching beta_schedule
    if hasattr(h_ind, '__len__') and len(h_ind) == len(beta_schedule_t):
            h_ind_is_schedule = True
            if not torch.is_tensor(h_ind):
                h_ind_t_schedule = torch.from_numpy(np.asarray(h_ind, dtype=np.float32)).to(device_)
            else:
                h_ind_t_schedule = h_ind.to(device_, dtype=torch.float32)

    num_systems, N = s0_all_t.shape
    s_all = torch.empty((num_systems, T, N), device=device_, dtype=torch.float32)
    s_all[:, 0, :] = s0_all_t

    cur = s0_all_t
    for t in range(1, T):
        beta_t = beta_schedule_t[t - 1]
        
        if h_ind_is_schedule:
            h_ind_val = h_ind_t_schedule[t - 1]
        else:
            h_ind_val = h_ind

        cur = pbit_step_both_batch(
            cur, J_t, h_t, beta_t,
            h_ind=h_ind_val,
            use_s0_for_ind=use_s0_for_ind,
            s0=s0_all_t if use_s0_for_ind else None,
            nbr_idx=nbr_idx,
            nbr_w=nbr_w,
        )
        s_all[:, t, :] = cur

    return s_all

def prob_trans_both(
    s_prev_batch: torch.Tensor,    # (B, N), ±1
    s_next: torch.Tensor,          # (N,), ±1
    beta: float,
    J: torch.Tensor | None,
    h: torch.Tensor,
    order=None,
    h_ind=1.0,
    use_s0_for_ind: bool = False,
    s0_batch: torch.Tensor | None = None,
    nbr_idx: torch.Tensor | None = None,
    nbr_w: torch.Tensor | None = None
) -> torch.Tensor:
    """
    Vectorized transition probability for batched spin systems.
    
    Parallelization strategy:
    - Parallel across batch (prior samples/candidates)
    - Sequential across spin update order (physics requirement)
    
    Returns probabilities shape (B,) for transition from each s_prev to s_next.
    The device is determined by the input tensors.
    """
    if order is None:
        N = s_prev_batch.shape[1]
        order = tuple(range(N))
    device_ = s_prev_batch.device
    B, N = s_prev_batch.shape

    # β, h_ind to tensors
    beta_t = torch.tensor(float(beta), device=device_, dtype=torch.float32)
    if not torch.is_tensor(h_ind):
        h_ind_t = torch.full((N,), float(h_ind), device=device_, dtype=torch.float32)
    elif h_ind.ndim == 0:
        h_ind_t = h_ind.expand(N).to(device_, dtype=torch.float32)
    else:
        h_ind_t = h_ind.to(device_, dtype=torch.float32)

    probs = torch.ones((B,), device=device_, dtype=torch.float32)

    s_temp = s_prev_batch.clone()
    base = s0_batch if use_s0_for_ind else s_prev_batch
    s_next_t = s_next.to(device_, dtype=torch.float32)

    # Use sparse neighbor list if provided
    use_sparse = (nbr_idx is not None and nbr_w is not None)

    for i in order:
        if use_sparse:
            # Sparse computation using neighbor list
            nbrs = nbr_idx[i]
            mask = nbrs >= 0
            if mask.any():
                nbrs_valid = nbrs[mask]
                w = nbr_w[i][mask]
                dep_field = (s_temp[:, nbrs_valid] * w).sum(dim=1) + h[i]
            else:
                dep_field = h[i].expand(B)
        elif J is not None:
            dep_field = torch.matmul(s_temp, J[i]) + h[i]   # (B,)
        else:
            dep_field = h[i].expand(B)

        ind_field = h_ind_t[i] * base[:, i]
        local = dep_field + ind_field
        p_i = 0.5 * (1.0 + s_next_t[i] * torch.tanh(beta_t * local))  # (B,)
        probs *= p_i
        s_temp[:, i] = s_next_t[i]

    return probs

def pbit_histogram_weighted_sampling_Nspin(
    s_t_np: np.ndarray,
    s0_np: np.ndarray,
    t: int,
    beta_schedule: np.ndarray,
    J_t: torch.Tensor | None,
    h_t: torch.Tensor,
    num_samples: int = 1000,
    order=None,
    h_ind: float | np.ndarray = 1.0,
    use_s0_for_ind: bool = False,
    nbr_idx: torch.Tensor | None = None,
    nbr_w: torch.Tensor | None = None,
):
    """
    Histogram-based weighted sampler for batched spin systems.
    
    Parallelization strategy:
    - Parallel across all prior samples (batch forward simulation)
    - Parallel across unique configurations for transition prob computation
    
    The device is determined by h_t's device.
    
    Returns:
        chosen configuration (NumPy array for compatibility)
    """
    device_ = h_t.device

    # Convert inputs to torch
    s_t = torch.from_numpy(s_t_np.astype(np.float32)).to(device_)
    s0 = torch.from_numpy(s0_np.astype(np.float32)).to(device_)

    # Handle h_ind schedule
    h_ind_is_schedule = False
    h_ind_t_schedule = None
    
    # Check if h_ind is a sequence with length matching beta_schedule
    if hasattr(h_ind, '__len__') and len(h_ind) == len(beta_schedule):
            h_ind_is_schedule = True
            if not torch.is_tensor(h_ind):
                h_ind_t_schedule = torch.from_numpy(np.asarray(h_ind, dtype=np.float32)).to(device_)
            else:
                h_ind_t_schedule = h_ind.to(device_, dtype=torch.float32)

    # 1. Generate prior samples s_{t-1} as a batch (num_samples, N)
    cur = s0.unsqueeze(0).expand(num_samples, -1).contiguous()  # (num_samples, N)
    for step in range(1, t):
        beta_step = float(beta_schedule[step-1])
        
        if h_ind_is_schedule:
            h_ind_val = h_ind_t_schedule[step - 1]
        else:
            h_ind_val = h_ind

        cur = pbit_step_both_batch(
            cur, J_t, h_t, beta_step,
            h_ind=h_ind_val,
            use_s0_for_ind=use_s0_for_ind,
            s0=s0.unsqueeze(0).expand(num_samples, -1) if use_s0_for_ind else None,
            nbr_idx=nbr_idx,
            nbr_w=nbr_w
        )
    samples = cur  # (num_samples, N)

    # 2. Unique configurations and counts
    unique, inverse, counts = torch.unique(
        samples, dim=0, return_inverse=True, return_counts=True
    )  # unique: (M,N), counts: (M,)

    # 3. Transition probabilities P(s_t | s_{t-1}) for all unique candidates
    beta_tminus1 = float(beta_schedule[t-1])
    
    if h_ind_is_schedule:
        h_ind_val_trans = h_ind_t_schedule[t - 1]
    else:
        h_ind_val_trans = h_ind

    trans = prob_trans_both(
        unique, s_t, beta_tminus1, J_t, h_t,
        order=order,
        h_ind=h_ind_val_trans,
        use_s0_for_ind=use_s0_for_ind,
        s0_batch=s0.unsqueeze(0).expand(unique.shape[0], -1) if use_s0_for_ind else None,
        nbr_idx=nbr_idx,
        nbr_w=nbr_w
    )  # (M,)

    # 4. Weights ∝ count * trans
    weights = counts.float() * trans
    weights_sum = weights.sum()
    if weights_sum <= 0:
        weights = torch.full_like(weights, 1.0 / len(weights))
    else:
        weights = weights / weights_sum

    # 5. Sample index and return configuration as NumPy ±1 ints (no .item() needed)
    chosen_idx = torch.multinomial(weights, num_samples=1).squeeze(0)
    chosen = unique[chosen_idx].cpu().numpy().astype(int)
    
    # Also return counts and weights for compatibility
    counts_np = counts.cpu().numpy()
    weights_np = weights.cpu().numpy()
    
    return chosen, counts_np, weights_np

In [ ]:
# === Minimal MLP that outputs per-site Bernoulli probs for s0 ===
class Diff_MLP(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 512, dropout: float = 0.0, use_layernorm: bool = False):
        super().__init__()
        layers = []
        
        # First layer
        layers.append(nn.Linear(input_size, hidden_size))
        if use_layernorm:
            layers.append(nn.LayerNorm(hidden_size))
        layers.append(nn.ReLU())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        
        # Second layer
        layers.append(nn.Linear(hidden_size, hidden_size))
        if use_layernorm:
            layers.append(nn.LayerNorm(hidden_size))
        layers.append(nn.ReLU())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        
        # Output layer
        layers.append(nn.Linear(hidden_size, input_size))
        layers.append(nn.Sigmoid())  # outputs in [0,1]
        
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

# === Helper to get \hat{s0} (±1) and probs from the model - FULLY TORCH ON GPU ===
@torch.no_grad()
def s0_hat_from_mlp_torch(model: nn.Module, s_t_pm1: torch.Tensor, sample: bool = True):
    """
    Fully GPU-based MLP inference helper (no CPU round-trip).
    
    Args:
        model: MLP model
        s_t_pm1: (B, N) or (N,) float32 on model device, values in {-1,+1}
        sample: if True, sample from Bernoulli; else use threshold
    
    Returns:
        s0_hat: (B, N) or (N,) tensor on device, values in {-1.0, +1.0}
        probs: (B, N) or (N,) tensor on device, values in [0,1]
    """
    # Handle both batched and single input
    squeeze_output = False
    if s_t_pm1.ndim == 1:
        s_t_pm1 = s_t_pm1.unsqueeze(0)
        squeeze_output = True
    
    x = (s_t_pm1 + 1.0) * 0.5              # map {-1,+1} to {0,1}
    probs = model(x)                       # (B, N), sigmoid output
    
    if sample:
        u = torch.rand_like(probs)
        s0_hat = torch.where(u < probs, 1.0, -1.0)
    else:
        s0_hat = torch.where(probs >= 0.5, 1.0, -1.0)
    
    if squeeze_output:
        s0_hat = s0_hat.squeeze(0)
        probs = probs.squeeze(0)
    
    return s0_hat, probs


In [ ]:
def pbit_reverse_diffusion_weighted_Nspin(
    s0_all: np.ndarray, sT_all: np.ndarray,
    beta_schedule: np.ndarray, J: np.ndarray, h: np.ndarray, T: int, rng,
    num_samples: int = 1000, order=None, h_ind: float | np.ndarray = 1.0,
    use_s0_for_ind: bool = False,
    use_mlp_estimator: bool = False,          
    mlp_model: nn.Module | None = None,       
    return_s0_predictions: bool = False,
    nbr_idx: torch.Tensor | None = None,
    nbr_w: torch.Tensor | None = None
):
    """
    Reverse trajectories (works on both CPU and GPU via device setting).
    
    Uses the same batched sampler regardless of device - torch functions work
    on both CPU and GPU automatically based on tensor device.
    
    If use_mlp_estimator=True, we replace the exact s0 with s0_hat predicted by mlp_model
    at each reverse step: s0_hat ~ p(s0 | s_t).
    
    If return_s0_predictions=True, also returns the s0_hat predictions at each timestep.
    """
    num_systems, _ = s0_all.shape
    out = np.empty((num_systems, T, J.shape[0]), dtype=int)
    out[:, T-1, :] = sT_all

    # Storage for s0 predictions if requested
    s0_predictions = None
    if return_s0_predictions and use_mlp_estimator:
        s0_predictions = np.empty((num_systems, T, J.shape[0]), dtype=int)

    if use_mlp_estimator and (mlp_model is None):
        raise ValueError("use_mlp_estimator=True but mlp_model is None.")

    # Setup tensors on device
    device_ = next(mlp_model.parameters()).device if mlp_model is not None else device
    J_t = torch.from_numpy(J.astype(np.float32)).to(device_) if np.count_nonzero(J) > 0 else None
    h_t = torch.from_numpy(h.astype(np.float32)).to(device_)
    s0_all_t = torch.from_numpy(s0_all.astype(np.float32)).to(device_)
    beta_schedule_t = torch.from_numpy(beta_schedule.astype(np.float32)).to(device_)
    
    # Handle h_ind schedule
    h_ind_is_schedule = hasattr(h_ind, '__len__') and len(h_ind) == len(beta_schedule)
    if h_ind_is_schedule:
        h_ind_t = torch.from_numpy(np.asarray(h_ind, dtype=np.float32)).to(device_)
    else:
        h_ind_t = float(h_ind)

    for k in range(num_systems):
        cur_t = torch.from_numpy(sT_all[k].astype(np.float32)).to(device_)  # (N,) on device
        out[k, T-1, :] = sT_all[k]

        for t in range(T-1, 0, -1):
            # Decide which s0 to use for the posterior at this step
            if use_mlp_estimator:
                s0_for_step_t, _ = s0_hat_from_mlp_torch(mlp_model, cur_t, sample=True)
                if return_s0_predictions:
                    s0_predictions[k, t, :] = s0_for_step_t.cpu().numpy().astype(int)
            else:
                s0_for_step_t = s0_all_t[k]  # (N,) on device

            # Weighted draw of s_{t-1} given s_t and s0_for_step
            cur_np = cur_t.cpu().numpy().astype(int)
            s0_for_step_np = s0_for_step_t.cpu().numpy().astype(int)
            
            # Get h_ind value for this timestep
            if h_ind_is_schedule:
                h_ind_val = h_ind_t[t-1]
            else:
                h_ind_val = h_ind_t
            
            cur_np, _, _ = pbit_histogram_weighted_sampling_Nspin(
                cur_np, s0_for_step_np, t, beta_schedule.astype(np.float32), J_t, h_t,
                num_samples=num_samples, order=order, h_ind=h_ind_val,
                use_s0_for_ind=use_s0_for_ind,
                nbr_idx=nbr_idx,
                nbr_w=nbr_w
            )
            cur_t = torch.from_numpy(cur_np.astype(np.float32)).to(device_)
            out[k, t-1, :] = cur_np

        if num_systems >= 10 and (k + 1) % (num_systems // 10) == 0:
            print(f"[reverse|p-bit] {100 * (k + 1) // num_systems}% systems completed")
        elif num_systems < 10 and (k + 1) % max(1, num_systems // 2) == 0:
            print(f"[reverse|p-bit] finished {k+1}/{num_systems} systems")
    
    if return_s0_predictions and s0_predictions is not None:
        return out, s0_predictions
    return out


In [ ]:
def plot_means_over_time(forward_means: np.ndarray, reverse_means: np.ndarray, theory_means: np.ndarray | None, title: str = "Spin Means"):
    """Plot ⟨s_i⟩ over time for forward, reverse, and (optional) probability-space theory."""
    T, N = forward_means.shape
    times = np.arange(T)
    colors = plt.cm.tab10(np.linspace(0, 1, min(N, 10)))
    plt.figure(figsize=(12, 6))
    for i in range(N):
        c = colors[i % 10]
        plt.plot(times, forward_means[:, i], '--', alpha=0.8, color=c, label=f's{i+1} (forward)' if i < 8 else None, marker='o', markersize=5)
        plt.plot(times, reverse_means[:, i], '-',  alpha=0.9, color=c, label=f's{i+1} (reverse)' if i < 8 else None, marker='o', markersize=5)
        if theory_means is not None:
            plt.plot(times, theory_means[:, i], ':',  lw=2, color=c, label=f's{i+1} (theory)' if i < 8 else None)
    plt.xlabel('t')
    plt.ylabel('⟨s_i⟩')
    plt.ylim(-1.05, 1.05)
    plt.grid(alpha=0.3)
    if N <= 8: plt.legend()
    plt.title(title)
    plt.tight_layout()
    plt.show()

def plot_pairwise_correlations(forward_pairs: np.ndarray, reverse_pairs: np.ndarray, theory_pairs: np.ndarray | None, choose=None, title: str = "Pairwise Correlations"):
    """Plot selected ⟨s_i s_j⟩ over time; choose is list of (i,j) with i<j. If None, show first 6 pairs."""
    T, N = forward_pairs.shape[0], forward_pairs.shape[1]
    all_pairs = [(i, j) for i in range(N) for j in range(i+1, N)]
    choose = choose or all_pairs[:6]
    times = np.arange(T)
    n = len(choose)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows), squeeze=False)
    for k, (i, j) in enumerate(choose):
        r, c = divmod(k, ncols)
        ax = axes[r, c]
        ax.plot(times, forward_pairs[:, i, j], '--', alpha=0.9, label='forward', marker='o', markersize=5)
        ax.plot(times, reverse_pairs[:, i, j], '-',  alpha=0.9, label='reverse', marker='o', markersize=5)
        if theory_pairs is not None:
            ax.plot(times, theory_pairs[:, i, j], ':', lw=2, label='theory')
        ax.set_title(f'⟨s{i+1}s{j+1}⟩')
        ax.set_ylim(-1.05, 1.05)
        ax.grid(alpha=0.3)
        if k == 0: ax.legend()
    for k in range(n, nrows*ncols):
        r, c = divmod(k, ncols)
        axes[r, c].axis('off')
    fig.suptitle(title)

    for ax_row in axes:
        for ax in ax_row:
            if ax.has_data():
                ax.set_xlabel('t')

    plt.tight_layout()
    plt.show()

In [ ]:
# ---- Config ----
grid_size = 50             # 2D grid size (50x50)
N = grid_size * grid_size  # Total number of spins (2500)
L = grid_size              # For compatibility
T = 101                     # Time steps

# ============================================================
# BETA_START: Initial state sampling temperature
# ============================================================
# VALID VALUES: Must be k/64 where k ∈ {0, 1, 2, ..., 80}
# Examples of valid beta_start values:
#   1.25      (k=80) - Most ordered, lowest energy states
#   1.0       (k=64) - Moderately ordered
#   0.75      (k=48) - Medium temperature
#   0.5       (k=32) - Higher temperature, less ordered
#   0.25      (k=16) - High temperature
#   0.0       (k=0)  - Infinite temperature, completely random
#   
# Other valid examples: 1.234375 (k=79), 0.625 (k=40), 0.375 (k=24), etc.
# ============================================================
beta_start = 0.453125          # Current: k=48, medium temperature initial states

num_systems = 10000          # Number of systems for training

# ---- 2D Ferromagnet FPGA data paths ----
trial_files_dir = r"L50_Results_noising_100000MCS_81beta_s46_nobias"

# ---- Set flags ----
use_TM = False
plot_exact = False
use_fwd_terminal = False

# ---- Unified diffusion scheduling ----
# Beta schedule matching FPGA data generation formula:
# k_vals = 80:-1:0 (MATLAB), beta_i = k / 64
# This gives 81 values: [1.25, 1.234375, 1.21875, ..., 0.015625, 0.0]
k_vals = np.arange(80, -1, -1)  # 80, 79, 78, ..., 1, 0 (81 values total)
full_beta_schedule = k_vals / 64  # Exact FPGA beta values: β_i = k/64

# Validate beta_start is one of the 81 valid FPGA values
valid_betas = set(full_beta_schedule)
if beta_start not in valid_betas:
    # Find closest valid beta
    closest_beta = full_beta_schedule[np.argmin(np.abs(full_beta_schedule - beta_start))]
    closest_k = int(round(closest_beta * 64))
    raise ValueError(
        f"\n{'='*70}\n"
        f"ERROR: Invalid beta_start = {beta_start}\n"
        f"{'='*70}\n"
        f"beta_start must be k/64 where k ∈ {{0, 1, 2, ..., 80}}\n\n"
        f"Valid values include:\n"
        f"  1.25      (k=80) - most ordered\n"
        f"  1.0       (k=64)\n"
        f"  0.75      (k=48)\n"
        f"  0.5       (k=32)\n"
        f"  0.25      (k=16)\n"
        f"  0.0       (k=0)  - random\n\n"
        f"Closest valid beta to {beta_start}: {closest_beta:.6f} (k={closest_k})\n"
        f"{'='*70}\n"
    )
beta_start_k = int(round(beta_start * 64))
print(f"✓ beta_start = {beta_start} is valid (k={beta_start_k})")

# Create beta schedule: FLAT/HIGH START, then SHARP DROP at end

t_norm = np.linspace(0, 1, T-1)
# beta_schedule = 0.0001 + (beta_start - 0.0001) * (1 - np.exp(t_norm - 1)) / (1 - np.exp(-1))  # T-1 diffusion steps
paramsched = 2.0
beta_schedule = 0.0001 + (beta_start - 0.0001) * (1 - np.exp(paramsched*t_norm - paramsched)) / (1 - np.exp(-paramsched))  # T-1 diffusion steps

eta_schedule = beta_to_eta_schedule(beta_schedule)
beta_sched_start = beta_schedule[0]  # First value of the schedule for checkpoint naming

h_ind = 0.0                # self-feedback strength
h_bias = 0.0               # NO external bias for 2D FM (data is nobias)

print("\nFPGA Beta Schedule (all 81 values):")
for i, (k, beta) in enumerate(zip(k_vals, full_beta_schedule)):
    if i < 5 or i > 75:  # Show first 5 and last 5
        print(f"  [{i:2d}] k={k:2d}, β = {beta:.6f}")
    elif i == 5:
        print(f"  ... (showing first 5 and last 5 only)")
print(f"\nSelected {len(beta_schedule)} beta values for T={T} timestep diffusion:")

# choose which state powers the independent term
use_s0_for_ind = False     # False => use s^{t-1}, True => use s^0

# turn off correlations for diffusion
turn_corr_off = False       # True => use J=0 for FORWARD diffusion

# set independent reverse sampler
set_ind_rev_sampler = False # True => use J=0 for REVERSE

# Transition-matrix / exact probability paths assume state-independent W.
if (use_TM or plot_exact):
    raise ValueError("Transition-matrix (use_TM) and exact (plot_exact) not supported when self-feedback is present.")

np.set_printoptions(precision=6, suppress=True)
print(f"\nGrid: {grid_size}x{grid_size}, N={N}, T={T}")
print(f"beta_start = {beta_start} (k={beta_start_k})")
print(f"Diffusion schedule: {T-1} steps from β={beta_start:.4f} down to β={beta_schedule[-1]:.4f}")
print(f"Selected beta_schedule:")
print(f"  {beta_schedule}")
print(f"h_ind={h_ind}, h_bias={h_bias}")

# Visualize beta schedule
plt.figure(figsize=(10, 5))
timesteps = np.arange(1, T)
plt.plot(timesteps, beta_schedule, 'o-', linewidth=2, markersize=4)
plt.xlabel('Diffusion Timestep t', fontsize=12)
plt.ylabel('β (Inverse Temperature)', fontsize=12)
plt.title('Beta Schedule: Noising Process', fontsize=13)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"\nBeta schedule visualization: β ranges from {beta_schedule[0]:.4f} to {beta_schedule[-1]:.4f}")

In [ ]:
### Load 2D Ferromagnet FPGA initial states and J matrix

import glob
import hdf5storage
from tqdm import tqdm
import os

# --- J matrix: nearest-neighbor 2D ferromagnet ---
def get_J_matrix(grid_size):
    """Generate nearest-neighbor ferromagnetic coupling for 2D grid."""
    size = grid_size * grid_size
    J = np.zeros((size, size))
    for row in range(grid_size):
        for col in range(grid_size):
            index = row * grid_size + col
            # Up neighbor
            if row > 0:
                neighbor = (row - 1) * grid_size + col
                J[index, neighbor] = 1
            # Down neighbor
            if row < grid_size - 1:
                neighbor = (row + 1) * grid_size + col
                J[index, neighbor] = 1
            # Left neighbor
            if col > 0:
                neighbor = row * grid_size + (col - 1)
                J[index, neighbor] = 1
            # Right neighbor
            if col < grid_size - 1:
                neighbor = row * grid_size + (col + 1)
                J[index, neighbor] = 1
    return J

J_original = get_J_matrix(grid_size)
print(f"[2D FM] Generated J matrix for {grid_size}x{grid_size} grid (nearest-neighbor coupling)")

# --- Build sparse neighbor list for GPU efficiency ---
print(f"Building sparse neighbor list from J matrix...")
nbr_idx, nbr_w = build_neighbor_list(J_original, device)
if nbr_idx is not None:
    print(f"Sparse neighbor list built: max degree = {nbr_idx.shape[1]}")
    print(f"Using sparse operations for local field computation (O(N*deg) vs O(N^2))")
else:
    print("J matrix is all zeros, no neighbor list needed")

# --- Bias vector (external field) ---
h = np.full(N, h_bias, dtype=float)

# --- Load FPGA initial states at specified beta ---
print(f"\n[2D FM] Loading initial states from FPGA data: {trial_files_dir}")
trial_files = sorted(glob.glob(os.path.join(trial_files_dir, "Trial_*.mat")))
num_files_available = len(trial_files)
print(f"Total trial files available: {num_files_available}")

# Check that we have enough files
if num_systems > num_files_available:
    raise ValueError(f"num_systems ({num_systems}) exceeds available files ({num_files_available}). "
                     f"Please reduce num_systems or add more trial files.")

print(f"Loading {num_systems} files (1 replica per file)")

# Calculate which beta index to load based on beta_start
# FPGA data structure: trial_results is (81, 1) with indices 0-80
# Index 0 → k=80, β=1.25
# Index 80 → k=0, β=0.0
# Formula: beta_index = 80 - k, where k = beta_start * 64
beta_index = 80 - beta_start_k
print(f"Loading states at beta_start={beta_start} (k={beta_start_k}, file index={beta_index})")

# Load 1 replica from each of num_systems files
initial_states = []
for file in tqdm(trial_files[:num_systems], desc=f"Loading states at β={beta_start}"):
    data = hdf5storage.loadmat(file)
    trial_cell = data['trial_results']  # shape: (81, 1) for 81 beta values
    # Get states at the specified beta index
    states_at_beta = trial_cell[beta_index, 0]  # shape: (replicas, 2500)
    # Take only the first replica from this file
    initial_states.append(states_at_beta[0])  # Shape: (2500,)

# Stack all initial states
states_all = np.stack(initial_states, axis=0)  # Shape: (num_systems, 2500)
np.set_printoptions(precision=1, suppress=True, linewidth=200)

S_loaded = states_all.shape[0]
print("\n2D FM Coupling Matrix J_original (showing first 10x10 block):")

print(f"\n[2D FM] Loaded {S_loaded} initial states (1 replica per file) at β={beta_start} (k={beta_start_k}) with N={N} spins")

print(J_original[:min(10,N), :min(10,N)])
print(f"These equilibrium states will be used as s0 for our own noising process")

In [ ]:
# ============================================================
# Skip forward diffusion and data loading if using saved model
# ============================================================
if USE_SAVED_MODEL:
    print("Skipping forward diffusion (USE_SAVED_MODEL=True)")
    print("Jump to the cell that loads the model for inference...")
    # We'll need to load J_original for reverse sampling, so do minimal loading:


In [ ]:
if not USE_SAVED_MODEL:
    print("USING 2D FERROMAGNET INITIAL STATES FROM FPGA DATA")

    # Use all loaded states (already have exactly num_systems states)
    s0_all = states_all.copy()
    print(f"[2D FM] Using all {num_systems} initial states (1 replica per file).")

    print(f"Starting forward diffusion process (our own noising)...")

    h_t = torch.from_numpy(h.astype(np.float32)).to(device)
    s0_all_t = torch.from_numpy(s0_all.astype(np.float32)).to(device)

    if turn_corr_off:
        print("=== Forward diffusion: INDEPENDENT (J=0) [turn_corr_off=True] ===")
        J_diffusion_t = None
    else:
        print("=== Forward diffusion: CORRELATED (J=J_original) [turn_corr_off=False] ===")
        J_diffusion_t = torch.from_numpy(J_original.astype(np.float32)).to(device)

    s_forward_all_t = pbit_forward_diffusion_all(
        s0_all_t, beta_schedule, J_diffusion_t, h_t, T,
        h_ind=h_ind, use_s0_for_ind=use_s0_for_ind,
        nbr_idx=nbr_idx if not turn_corr_off else None,
        nbr_w=nbr_w if not turn_corr_off else None
    )

    s_forward_all = s_forward_all_t.sign().cpu().numpy().astype(np.int8)
    print(f"Using unified forward diffusion on device: {device}")

    print("J_original (for initial sampling and energy) =")
    print(J_original[:min(10,N), :min(10,N)])
    print(J_original.shape)
else:
    print("Skipping forward diffusion (USE_SAVED_MODEL=True)")

In [ ]:
if not USE_SAVED_MODEL:
    # === Visualize forward diffusion process ===
    from matplotlib.colors import ListedColormap
    
    N_fwd_systems_plot = 10  # Number of systems to visualize
    selected_fwd_systems = range(min(N_fwd_systems_plot, num_systems))

    print(f"\n=== Visualizing forward diffusion for {len(selected_fwd_systems)} systems ===")

    # Blue for -1, Red for +1
    spin_cmap = ListedColormap(['blue', 'red'])
    
    # Fixed timesteps: 0, 1, 11, 22, 33, 44, 55, 66, 77, 88, 99, 100
    plot_timesteps = np.array([0, 1, 11, 22, 33, 44, 55, 66, 77, 88, 99, 100])
    plot_timesteps = plot_timesteps[plot_timesteps < T]
    
    # Plot forward states for selected systems
    for sys_idx in selected_fwd_systems:
        fig, axes = plt.subplots(1, len(plot_timesteps), figsize=(2*len(plot_timesteps), 2.5))
        if len(plot_timesteps) == 1:
            axes = [axes]
        
        fig.suptitle(f'Forward Diffusion for System {sys_idx}\n← Clean (t=0) ... Noisy (t={T-1}) →', 
                     fontsize=12, y=1.02)
        
        # Plot forward states at selected timesteps (left to right)
        for idx, t in enumerate(plot_timesteps):
            s_t = s_forward_all[sys_idx, t, :].reshape(grid_size, grid_size)
            
            # Convert bipolar (-1,+1) to binary (0,1) for colormap
            s_t_binary = (s_t + 1) / 2
            
            ax = axes[idx]
            ax.imshow(s_t_binary, cmap=spin_cmap, vmin=0, vmax=1, interpolation='nearest')
            ax.set_title(f't={t}\nβ={beta_schedule[min(t, len(beta_schedule)-1)]:.2f}', fontsize=9)
            ax.axis('off')
        
        plt.tight_layout()
        fig.savefig(f"plot_2_fwd_trajs_beta_{beta_start:.6f}_sys_{sys_idx:03d}-corr.pdf",
            format="pdf", bbox_inches="tight")
        plt.show()

    print("\nDone plotting forward diffusion trajectories.")
else:
    print("Skipping forward diffusion visualization (USE_SAVED_MODEL=True)")

In [ ]:
# === Visualize Energy and Magnetization for Forward Process Only ===

def _energy_per_spin_batch(states: torch.Tensor, J: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
    """
    Energy computation for batched spin systems.
    
    Parallelization strategy:
    - Dense matrix operations fully parallelized
    - Batch dimension (systems) processed in parallel
    
    states: (B, N), ±1
    J:      (N, N)
    h:      (N,)
    returns: (B,) energy per spin
    
    Device is determined by input tensors - same code works on CPU or GPU.
    """
    quad = -0.5 * torch.einsum('bi,ij,bj->b', states, J, states)
    lin  = - torch.matmul(states, h)
    E = quad + lin
    return E / states.shape[1]

def _magnetization_per_spin_batch(states: torch.Tensor) -> torch.Tensor:
    """
    Magnetization computation for batched spin systems.
    
    Trivially parallelized reduction across spins for each system.
    Device is determined by input tensor.
    """
    return states.mean(dim=1)

if not USE_SAVED_MODEL:
    print(f"\nComputing energy/magnetization for forward process (n={num_systems})...")
    print(f"Using device: {device}")

    J_energy = torch.from_numpy(J_original.astype(np.float32)).to(device)
    h_energy = torch.from_numpy(h.astype(np.float32)).to(device)
    s_forward_energy = torch.from_numpy(s_forward_all.astype(np.float32)).to(device)

    Eps_forward = np.empty((T, num_systems))
    M_forward   = np.empty((T, num_systems))

    for t in range(T):
        Ef_t = _energy_per_spin_batch(s_forward_energy[:, t, :], J_energy, h_energy).cpu().numpy()
        Mf_t = _magnetization_per_spin_batch(s_forward_energy[:, t, :]).cpu().numpy()
        Eps_forward[t] = Ef_t
        M_forward[t]   = Mf_t

    # Compute means
    mean_Eps_forward = Eps_forward.mean(axis=1)
    mean_absM_forward = np.abs(M_forward).mean(axis=1)
    
    times = np.arange(T)

    plt.figure(figsize=(11, 4.5))

    # ---- Energy per spin evolution ----
    plt.subplot(1, 2, 1)
    plt.plot(times, mean_Eps_forward, 'o--', label='forward', color='tab:blue')
    plt.xlabel('Diffusion timestep t')
    plt.ylabel('⟨E/N⟩')
    plt.title('Mean energy per spin vs time (Forward Only)')
    plt.grid(alpha=0.3)
    plt.legend()

    # ---- |Magnetization| evolution ----
    plt.subplot(1, 2, 2)
    plt.plot(times, mean_absM_forward, 'o--', label='forward', color='tab:green')
    plt.xlabel('Diffusion timestep t')
    plt.ylabel('⟨|m|⟩')
    plt.title('Mean Absolute magnetization vs time (Forward Only)')
    plt.grid(alpha=0.3)
    plt.legend()

    plt.suptitle(f'Forward Process Evolution (L={int(L)})\nForward: ALL training data (n={num_systems})', y=1.05)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping forward process energy/magnetization plot (USE_SAVED_MODEL=True)")

In [ ]:
# ============================================================
# Visualize Spin Evolution for First 10 Chains
# ============================================================
if not USE_SAVED_MODEL:
    from matplotlib.colors import ListedColormap, BoundaryNorm

    num_chains_to_plot = 10
    num_chains_to_plot = min(num_chains_to_plot, num_systems)

    print(f"Visualizing spin evolution for first {num_chains_to_plot} chains...")

    # 2x5 grid
    rows = 2
    cols = 5
    fig, axes = plt.subplots(rows, cols, figsize=(20, 8))
    axes = axes.flatten()

    # Colormap setup (Blue for -1 -> 0, Yellow for +1 -> 1)
    cmap = ListedColormap(["blue", "yellow"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5], cmap.N)

    for i in range(num_chains_to_plot):
        ax = axes[i]
        
        # Get trajectory for system i: shape (T, N)
        # s_forward_all is (num_systems, T, N)
        trajectory = s_forward_all[i] 
        
        # Transpose for (Spins x Time) -> (N, T)
        spins_vs_time = trajectory.T
        
        # Map -1 to 0, 1 to 1 for plotting with the specific colormap
        data = np.where(spins_vs_time == -1, 0, 1)
        
        im = ax.imshow(
            data,
            aspect="auto",
            interpolation="nearest",
            cmap=cmap,
            norm=norm,
            origin="lower",
            extent=[0, T, 0, N] # x from 0 to T, y from 0 to N
        )
        
        ax.set_title(f"Chain {i}")
        
        # Set labels
        if i >= 5:
            ax.set_xlabel("Time step (t)")
        if i % 5 == 0:
            ax.set_ylabel("Spin index")

    for i in range(num_chains_to_plot, len(axes)):
        axes[i].axis('off')

    plt.suptitle(f"Spin Evolution vs Time (First {num_chains_to_plot} Chains)", fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping spin evolution plot (USE_SAVED_MODEL=True)")

In [ ]:
if not USE_SAVED_MODEL:
    # === ADD: Train a simple MLP to estimate s0 from s_t ===
    # Build dataset: inputs X are s_forward_all[:,1:,:], targets y are s0_all
    num_systems, T_local, N_local = s_forward_all.shape
    assert N_local == N and T_local == T

    # ---- Test-Train split ----
    perm_sys = rng.permutation(num_systems)
    n_train_sys = max(1, int(0.8 * num_systems))
    train_sys = perm_sys[:n_train_sys]
    val_sys   = perm_sys[n_train_sys:]

    X_train = ((s_forward_all[train_sys, 1:, :] + 1.0) * 0.5).reshape(-1, N).astype(np.float32)
    y_train = np.repeat(((s0_all[train_sys] == 1).astype(np.float32)), repeats=(T - 1), axis=0)

    X_val   = ((s_forward_all[val_sys,   1:, :] + 1.0) * 0.5).reshape(-1, N).astype(np.float32)
    y_val   = np.repeat(((s0_all[val_sys]   == 1).astype(np.float32)), repeats=(T - 1), axis=0)

    print(f"Dataset split: {len(train_sys)} train systems ({len(X_train)} samples), "
          f"{len(val_sys)} val systems ({len(X_val)} samples)")

    X_train = torch.from_numpy(X_train); y_train = torch.from_numpy(y_train)
    X_val   = torch.from_numpy(X_val);   y_val   = torch.from_numpy(y_val)

    EPOCHS = 4500
    PATIENCE = EPOCHS // 25
    hidden_size = 1024
    dropout = 0.0
    use_layernorm = False
    LR = 1e-6
    weight_decay = 0.0
    batch_size = 512
    use_cosine_annealing = False

    # Generate descriptive checkpoint filename based on config (without val_loss yet)
    if CHECKPOINT_PATH is None:
        # Handle h_ind formatting (it might be an array/schedule now)
        if hasattr(h_ind, "__len__") and not isinstance(h_ind, str):
             h_ind_str = f"Sched{h_ind[0]:.2f}to{h_ind[-1]:.2f}"
        else:
             h_ind_str = f"{h_ind:.2f}"

        # Create a base filename with all key parameters (val_loss added after training)
        ckpt_base = (
            f"ckpt_2dfm_"
            f"b{beta_start:.2f}_"
            f"bss{beta_sched_start:.2f}_"
            f"T{T}_"
            f"n{num_systems}_"
            f"hi{h_ind_str}_"
            f"us0{'T' if use_s0_for_ind else 'F'}_"
            f"tc{'T' if turn_corr_off else 'F'}_"
            f"ep{EPOCHS}_"
            f"pat{PATIENCE}_"
            f"h{hidden_size}_"
            f"d{dropout:.2f}_"
            f"ln{'T' if use_layernorm else 'F'}_"
            f"lr{LR:.0e}_"
            f"wd{weight_decay:.0e}_"
            f"bs{batch_size}_"
            f"ca{'T' if use_cosine_annealing else 'F'}"
        )
        CHECKPOINT_PATH_BASE = ckpt_base
        CHECKPOINT_PATH = f"{ckpt_base}_TEMP.pth"
        print(f"Checkpoint will be saved with auto-generated name (val_loss added after training)")
    else:
        CHECKPOINT_PATH_BASE = None
        print(f"Using specified checkpoint path: {CHECKPOINT_PATH}")

    print("Training neural network with batching...")
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=batch_size, shuffle=False)
    model = Diff_MLP(input_size=N, hidden_size=hidden_size, dropout=dropout, use_layernorm=use_layernorm).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=weight_decay)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS) if use_cosine_annealing else None

    loss_history = []
    val_loss_history = []

    best_val_loss = float('inf')
    best_model_state = None
    best_epoch = -1            
    best_train_loss = None     
    patience_counter = 0

    for epoch in range(EPOCHS):
        model.train()
        running = 0.0
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            out = model(batch_X)
            loss = criterion(out, batch_y)
            loss.backward()
            optimizer.step()
            running += loss.item()
        train_loss = running / max(1, len(train_loader))

        model.eval()
        v_running = 0.0
        with torch.no_grad():
            for vX, vY in val_loader:
                vX = vX.to(device)
                vY = vY.to(device)
                
                vout = model(vX)
                vloss = criterion(vout, vY)
                v_running += vloss.item()
        val_loss = v_running / max(1, len(val_loader))

        loss_history.append(train_loss)
        val_loss_history.append(val_loss)

        if scheduler is not None:
            scheduler.step()

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            best_epoch = epoch          # Track best epoch
            best_train_loss = train_loss
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 100 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch:3d} | train {train_loss:.5f} | val {val_loss:.5f} | lr {current_lr:.2e} | patience {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            print(f"Best validation loss: {best_val_loss:.5f}")
            break

    if best_model_state is not None:
        import copy
        best_model_state = copy.deepcopy(model.state_dict())
        print(f"Restored model with best validation loss: {best_val_loss:.5f}")
    else:
        print(f"Final epoch {epoch:3d} | train {train_loss:.5f} | val {val_loss:.5f}")

    # --- Save checkpoint of the best model for later use ---
    if best_model_state is not None:
        checkpoint = {
            # Model architecture and weights
            "model_state_dict": model.state_dict(), 
            "input_size": N,
            "hidden_size": hidden_size,
            "dropout": dropout,
            "use_layernorm": use_layernorm,
            
            # Training results
            "epoch": best_epoch,
            "best_train_loss": best_train_loss,
            "best_val_loss": best_val_loss,
            
            # Training hyperparameters
            "EPOCHS": EPOCHS,
            "PATIENCE": PATIENCE,
            "LR": LR,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "use_cosine_annealing": use_cosine_annealing,
            
            # Physics parameters
            "beta_schedule": beta_schedule,
            "beta_start": beta_start,
            "beta_sched_start": beta_sched_start,
            "N": N,
            "T": T,
            "num_systems": num_systems,
            "h_ind": h_ind,
            "use_s0_for_ind": use_s0_for_ind,
            "turn_corr_off": turn_corr_off,
            "set_ind_rev_sampler": set_ind_rev_sampler,
        }
        
        torch.save(checkpoint, CHECKPOINT_PATH)
        
        if CHECKPOINT_PATH_BASE is not None:
            import os
            final_path = f"{CHECKPOINT_PATH_BASE}_vl{best_val_loss:.5f}.pth"
            os.rename(CHECKPOINT_PATH, final_path)
            CHECKPOINT_PATH = final_path
            print(f"Saved best model checkpoint to {CHECKPOINT_PATH}")
        else:
            print(f"Saved best model checkpoint to {CHECKPOINT_PATH}")
else:
    print(f"\nLoading model from checkpoint for inference...")
    
    # Model was already loaded to CPU in the early cell, now move to device
    model = Diff_MLP(
        input_size=checkpoint["input_size"],
        hidden_size=checkpoint["hidden_size"],
        dropout=checkpoint["dropout"],
        use_layernorm=checkpoint["use_layernorm"],
    ).to(device)
    
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    
    print(f"Model loaded and ready for inference on device: {device}")
    print(f"  Loaded from epoch {checkpoint['epoch']}, best val loss = {checkpoint['best_val_loss']:.5f}")

In [ ]:
# Plot loss vs epochs (only if we just trained)
if not USE_SAVED_MODEL:
    plt.figure(figsize=(6,4))
    plt.plot(loss_history, label='Train Loss')
    plt.plot(val_loss_history, label='Validation Loss')
    plt.axvline(best_epoch, color='red', linestyle='--', linewidth=1.0, label='Best epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss vs Epochs')
    plt.legend()
    plt.savefig("plot_2_losscurve_0p453125_corr.pdf", format="pdf", bbox_inches="tight")
    plt.show()
else:
    print("Skipping loss plot (model loaded from checkpoint)")

In [ ]:
### Conditional reverse diffusion based on set_ind_rev_sampler flag

model.eval()
num_systems_inference = 100
num_samples = 10

if USE_SAVED_MODEL:
    # When loading from checkpoint, sample fresh initial states for inference
    print("Inference mode: Sampling fresh initial states from loaded data...")
    num_systems_inference = min(num_systems_inference, states_all.shape[0])
    inference_indices = rng.choice(states_all.shape[0], size=num_systems_inference, replace=False)
    s0_inference = states_all[inference_indices]
    
    # Generate random terminal states (no forward diffusion was run)
    sT_inference = rng.choice([-1, 1], size=(num_systems_inference, N)).astype(int)
    print(f"Using {num_systems_inference} fresh states for inference")
else:
    # Standard mode: use data from forward diffusion
    if num_systems_inference > num_systems:
        raise ValueError(f"num_systems_inference ({num_systems_inference}) cannot exceed num_systems ({num_systems})")

    # Select a subset of s0_all for inference
    inference_indices = rng.choice(num_systems, size=num_systems_inference, replace=False)
    s0_inference = s0_all[inference_indices]

    if use_fwd_terminal:
        sT_inference = s_forward_all[inference_indices, -1, :]
    else:
        sT_inference = rng.choice([-1, 1], size=(num_systems_inference, N)).astype(int)

if set_ind_rev_sampler:
    # Single-mode: INDEPENDENT reverse (J=0)
    print("=== Reverse diffusion: INDEPENDENT (J=0) [set_ind_rev_sampler=True] ===")
    s_reverse_all, s0_predictions = pbit_reverse_diffusion_weighted_Nspin(
        s0_inference, sT_inference, beta_schedule, np.zeros((N,N)), h, T, rng,
        num_samples=num_samples, order=None, h_ind=h_ind,
        use_s0_for_ind=use_s0_for_ind,
        use_mlp_estimator=True, mlp_model=model, return_s0_predictions=True,
        nbr_idx=None, nbr_w=None
    )
    print("Using independent reverse (J=0) for sampling.")
else:
    # Single-mode: CORRELATED reverse (J=J_original)
    print("=== Reverse diffusion: CORRELATED (J=J_original) [set_ind_rev_sampler=False] ===")
    s_reverse_all, s0_predictions = pbit_reverse_diffusion_weighted_Nspin(
        s0_inference, sT_inference, beta_schedule, J_original, h, T, rng,
        num_samples=num_samples, order=None, h_ind=h_ind,
        use_s0_for_ind=use_s0_for_ind,
        use_mlp_estimator=True, mlp_model=model, return_s0_predictions=True,
        nbr_idx=nbr_idx, nbr_w=nbr_w
    )
    print("Using correlated reverse (J=J_original) for sampling.")

print(f"Performed inference on {num_systems_inference} systems")

# ---- Averages and correlations ----
# Compare forward data (if available) vs generated samples
if not USE_SAVED_MODEL:
    print(f"\nComputing forward statistics from ALL {num_systems} training samples...")
    forward_means, forward_pairs = avg_trajectory_and_pairs(s_forward_all)
    print(f"Computing reverse statistics from {num_systems_inference} generated samples...")
    reverse_means, reverse_pairs = avg_trajectory_and_pairs(s_reverse_all)

    # ---- Plots (minimal) ----
    plot_means_over_time(forward_means, reverse_means, None, 
                         title=f"{N}-spin ⟨s_i⟩ (forward n={num_systems}, reverse n={num_systems_inference})")
    plot_pairwise_correlations(forward_pairs, reverse_pairs, None, 
                           title=f"{N}-spin ⟨s_i s_j⟩ (forward n={num_systems}, reverse n={num_systems_inference})")
else:
    # Only compute reverse statistics when using saved model
    print(f"\nComputing reverse statistics from {num_systems_inference} generated samples...")
    reverse_means, reverse_pairs = avg_trajectory_and_pairs(s_reverse_all)
    print("Skipping forward/reverse comparison plots (no forward data available)")

In [ ]:
# === Visualize reverse diffusion process ===
from matplotlib.colors import ListedColormap

N_rev_systems_plot = 10  # Number of systems to visualize
selected_rev_systems = range(min(N_rev_systems_plot, num_systems_inference))

print(f"\n=== Visualizing reverse diffusion for {len(selected_rev_systems)} systems ===")

# Blue for -1, Red for +1
spin_cmap = ListedColormap(['blue', 'red'])

# Fixed timesteps: 0, 1, 11, 22, 33, 44, 55, 66, 77, 88, 99, 100
plot_timesteps = np.array([0, 1, 11, 22, 33, 44, 55, 66, 77, 88, 99, 100])
plot_timesteps = plot_timesteps[plot_timesteps < T]

# Plot reverse states for selected systems
for sys_idx in selected_rev_systems:
    fig, axes = plt.subplots(1, len(plot_timesteps), figsize=(2*len(plot_timesteps), 2.5))
    if len(plot_timesteps) == 1:
        axes = [axes]
    
    fig.suptitle(f'Reverse Diffusion for System {sys_idx}\n← Noisy (t=0) ... Clean (t={T-1}) →', 
                 fontsize=12, y=1.02)
    
    # Plot reverse states at selected timesteps (left to right: noisy -> clean)
    for idx, t in enumerate(plot_timesteps):
        s_t = s_reverse_all[sys_idx, t, :].reshape(grid_size, grid_size)
        
        # Convert bipolar (-1,+1) to binary (0,1) for colormap
        s_t_binary = (s_t + 1) / 2
        
        ax = axes[idx]
        ax.imshow(s_t_binary, cmap=spin_cmap, vmin=0, vmax=1, interpolation='nearest')
        ax.set_title(f't={t}\nβ={beta_schedule[min(t, len(beta_schedule)-1)]:.2f}', fontsize=9)
        ax.axis('off')
    
    plt.tight_layout()
    fig.savefig(f"plot_2_rev_trajs_beta_{beta_start:.6f}_sys_{sys_idx:03d}-corr.pdf",
            format="pdf", bbox_inches="tight")
    plt.show()

print("\nDone plotting reverse diffusion trajectories.")

In [ ]:
# === Visualize MLP s0 predictions and actual reverse states ===
from matplotlib.colors import ListedColormap

N_s0_for_step_plot = 5  # Number of systems to visualize

# Select systems to plot (first N_s0_for_step_plot from inference set)
selected_systems = range(min(N_s0_for_step_plot, num_systems_inference))

print(f"\n=== Visualizing s0_hat predictions and actual reverse states for {len(selected_systems)} systems ===")

# Blue for -1, Red for +1
spin_cmap = ListedColormap(['blue', 'red'])

# Plot s0_hat predictions and actual s_t states for selected systems
for sys_idx in selected_systems:
    # Determine which timesteps to plot
    if T > 11:
        num_plots = 10
        plot_timesteps = np.linspace(0, T-1, num_plots, dtype=int)
    else:
        plot_timesteps = np.arange(T)
    
    # Create two rows: top row for MLP s₀ predictions, bottom row for actual reverse states
    fig, axes = plt.subplots(2, len(plot_timesteps), figsize=(2*len(plot_timesteps), 5))
    if len(plot_timesteps) == 1:
        axes = axes.reshape(2, 1)
    
    fig.suptitle(f'Inference System {sys_idx}: MLP s₀ Predictions (top) vs Actual Reverse States (bottom)\n← Noisy (t={T-1}) ... Clean (t=0) →', 
                 fontsize=12, y=0.98)
    
    # Top row: MLP s₀ predictions
    for col_idx, t in enumerate(plot_timesteps):
        if t == 0:
            # Show actual s0 (ground truth at t=0)
            s0_true = s0_inference[sys_idx].reshape(grid_size, grid_size)
            s0_true_binary = (s0_true + 1) / 2
            
            ax = axes[0, col_idx]
            ax.imshow(s0_true_binary, cmap=spin_cmap, vmin=0, vmax=1, interpolation='nearest')
            ax.set_title(f'True s₀ (t=0)', fontsize=9, color='green')
            ax.axis('off')
        else:
            # MLP prediction at this timestep
            s0_pred = s0_predictions[sys_idx, t, :].reshape(grid_size, grid_size)
            s0_pred_binary = (s0_pred + 1) / 2
        
            ax = axes[0, col_idx]
            ax.imshow(s0_pred_binary, cmap=spin_cmap, vmin=0, vmax=1, interpolation='nearest')
            ax.set_title(f'MLP ŝ₀ @t={t}', fontsize=9)
            ax.axis('off')
    
    # Bottom row: Actual reverse states
    for col_idx, t in enumerate(plot_timesteps):
        s_t = s_reverse_all[sys_idx, t, :].reshape(grid_size, grid_size)
        s_t_binary = (s_t + 1) / 2
        
        ax = axes[1, col_idx]
        ax.imshow(s_t_binary, cmap=spin_cmap, vmin=0, vmax=1, interpolation='nearest')
        ax.set_title(f'Reverse s_{t}', fontsize=9, color='blue')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

print("\nDone plotting individual system predictions and reverse states.")

In [ ]:
# --- Energy-per-spin & Magnetization distributions per timestep (forward vs reverse) ---

# Compute per-timestep arrays using J_original for energy calculations
if not USE_SAVED_MODEL:
    # Compare ALL training data vs generated samples
    print(f"\nComputing energy/magnetization for ALL {num_systems} training samples vs {num_systems_inference} generated samples...")
    print(f"Using device: {device}")

    J_energy = torch.from_numpy(J_original.astype(np.float32)).to(device)
    h_energy = torch.from_numpy(h.astype(np.float32)).to(device)

    s_forward_energy = torch.from_numpy(s_forward_all.astype(np.float32)).to(device)
    s_reverse_energy = torch.from_numpy(s_reverse_all.astype(np.float32)).to(device)

    Eps_forward = np.empty((T, num_systems))
    Eps_reverse = np.empty((T, num_systems_inference))
    M_forward   = np.empty((T, num_systems))
    M_reverse   = np.empty((T, num_systems_inference))

    for t in range(T):
        Ef_t = _energy_per_spin_batch(s_forward_energy[:, t, :], J_energy, h_energy).cpu().numpy()
        Er_t = _energy_per_spin_batch(s_reverse_energy[:, t, :], J_energy, h_energy).cpu().numpy()
        Mf_t = _magnetization_per_spin_batch(s_forward_energy[:, t, :]).cpu().numpy()
        Mr_t = _magnetization_per_spin_batch(s_reverse_energy[:, t, :]).cpu().numpy()

        Eps_forward[t] = Ef_t
        Eps_reverse[t] = Er_t
        M_forward[t]   = Mf_t
        M_reverse[t]   = Mr_t
else:
    # Inference mode: only compute reverse energies
    print(f"\nComputing energy/magnetization for {num_systems_inference} generated samples...")
    print(f"Using device: {device}")

    J_energy = torch.from_numpy(J_original.astype(np.float32)).to(device)
    h_energy = torch.from_numpy(h.astype(np.float32)).to(device)
    s_reverse_energy = torch.from_numpy(s_reverse_all.astype(np.float32)).to(device)

    Eps_reverse = np.empty((T, num_systems_inference))
    M_reverse   = np.empty((T, num_systems_inference))

    # Compute only reverse statistics
    for t in range(T):
        Er_t = _energy_per_spin_batch(s_reverse_energy[:, t, :], J_energy, h_energy).cpu().numpy()
        Mr_t = _magnetization_per_spin_batch(s_reverse_energy[:, t, :]).cpu().numpy()
        Eps_reverse[t] = Er_t
        M_reverse[t]   = Mr_t

In [ ]:
# --- Mean Energy and Mean |Magnetization| over timesteps (forward vs reverse) ---

if not USE_SAVED_MODEL:
    # Compute mean energy per spin and mean |magnetization| across systems
    mean_Eps_forward = Eps_forward.mean(axis=1)
    mean_Eps_reverse = Eps_reverse.mean(axis=1)
    mean_absM_forward = np.abs((M_forward).mean(axis=1))
    mean_absM_reverse = np.abs((M_reverse).mean(axis=1))

    times = np.arange(T)

    plt.figure(figsize=(11, 4.5))

    # ---- Energy per spin evolution ----
    plt.subplot(1, 2, 1)
    plt.plot(times, mean_Eps_forward, 'o--', label='forward', color='tab:blue')
    plt.plot(times[1:], mean_Eps_reverse[1:], 'o-',  label='reverse', color='tab:orange')
    plt.xlabel('Diffusion timestep t')
    plt.ylabel('⟨E/N⟩')
    plt.title('Mean energy per spin vs time')
    plt.grid(alpha=0.3)
    plt.legend()

    # ---- |Magnetization| evolution ----
    plt.subplot(1, 2, 2)
    plt.plot(times, mean_absM_forward, 'o--', label='forward', color='tab:green')
    plt.plot(times[1:], mean_absM_reverse[1:], 'o-',  label='reverse', color='tab:red')
    plt.ylim(-0.05, 0.6)
    plt.xlabel('Diffusion timestep t')
    plt.ylabel('|⟨m⟩|')
    plt.title('Absolute Mean magnetization vs time')
    plt.grid(alpha=0.3)
    plt.legend()

    plt.suptitle(f'Evolution of mean quantities during diffusion 2D Ferromagnet L={int(L)}\nForward: ALL training data (n={num_systems}), Reverse: generated samples (n={num_systems_inference})\nChains={num_samples}, Correlated FWD: {not turn_corr_off}, Correlated reverse sampler: {not set_ind_rev_sampler}', y=1.05)
    plt.tight_layout()
    plt.savefig("plot_2_forwardandrev_trajs_0p453125_only_corr_absofmean.pdf", format="pdf", bbox_inches="tight")
    plt.show()
else:
    print("Skipping forward/reverse energy comparison plot (no forward data available)")

In [ ]:
# ---- Print selected timestep statistics ----

timesteps_to_check = [1, 11, 33, 55, 100]

print("\n===== Mean quantities at selected timesteps =====")

for t in timesteps_to_check:
    if t < T:
        print(f"\nTimestep t = {t}")
        print(f"  Mean E/N (forward)  : {mean_Eps_forward[t]:.8f}")
        print(f"  Mean E/N (reverse)  : {mean_Eps_reverse[t]:.8f}")
        print(f"  Mean |m| (forward)  : {mean_absM_forward[t]:.8f}")
        print(f"  Mean |m| (reverse)  : {mean_absM_reverse[t]:.8f}")
    else:
        print(f"\nTimestep t = {t} is out of range (T = {T})")

# ---- Specific system values ----
system_index = 5   # system number 6 (0-based indexing)
t_specific = 1

print("\n===== System 6 values at timestep t = 1 =====")

if t_specific < T and system_index < Eps_forward.shape[1]:
    print(f"Energy per spin (forward)  : {Eps_forward[t_specific, system_index]:.8f}")
    print(f"Energy per spin (reverse)  : {Eps_reverse[t_specific, system_index]:.8f}")
    
    print(f"Absolute magnetization (forward) : {abs(M_forward[t_specific, system_index]):.8f}")
    print(f"Absolute magnetization (reverse) : {abs(M_reverse[t_specific, system_index]):.8f}")
else:
    print("Requested timestep or system index out of range.")


In [ ]:
# ---- Mean trajectories (needed for later plots) ----
mean_Eps_reverse  = Eps_reverse.mean(axis=1)          # shape (T,)
mean_absM_reverse = np.abs(M_reverse).mean(axis=1)    # shape (T,)

# === Compute averaged statistics for s0 predictions ===
print("\n=== Computing energy and magnetization for MLP s0 predictions ===")
print(f"Using device: {device}")

# Compute energy per spin and magnetization for s0 predictions at each timestep
# Initialize with NaN for all timesteps
Eps_s0_pred = np.full((T, num_systems_inference), np.nan)
M_s0_pred   = np.full((T, num_systems_inference), np.nan)

if 'J_energy' not in locals():  # If not already defined from previous section
    J_energy = torch.from_numpy(J_original.astype(np.float32)).to(device)
    h_energy = torch.from_numpy(h.astype(np.float32)).to(device)
s0_pred_t = torch.from_numpy(s0_predictions.astype(np.float32)).to(device)

# For timesteps where we have predictions (t=1 to T-1)
for t in range(1, T):
    Eps_s0_pred[t] = _energy_per_spin_batch(s0_pred_t[:, t, :], J_energy, h_energy).cpu().numpy()
    M_s0_pred[t]   = _magnetization_per_spin_batch(s0_pred_t[:, t, :]).cpu().numpy()

# Compute mean values across systems (axis=1)
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    mean_Eps_s0_pred = np.nanmean(Eps_s0_pred, axis=1)
    mean_absM_s0_pred = np.abs(np.nanmean(M_s0_pred, axis=1))

# Also compute for true s0 for comparison
if not USE_SAVED_MODEL:
    # Use training data s0_all
    s0_all_t = torch.from_numpy(s0_all.astype(np.float32)).to(device)
    Eps_s0_true_all = _energy_per_spin_batch(s0_all_t, J_energy, h_energy).cpu().numpy().mean()
    M_s0_true_all = np.abs(_magnetization_per_spin_batch(s0_all_t).cpu().numpy().mean())
    print(f"True s₀ (ALL training data, n={num_systems}): E/N = {Eps_s0_true_all:.4f}, |⟨m⟩| = {M_s0_true_all:.4f}")
else:
    # Use inference s0_inference as reference
    s0_inf_t = torch.from_numpy(s0_inference.astype(np.float32)).to(device)
    Eps_s0_true_all = _energy_per_spin_batch(s0_inf_t, J_energy, h_energy).cpu().numpy().mean()
    M_s0_true_all = np.abs(_magnetization_per_spin_batch(s0_inf_t).cpu().numpy().mean())
    print(f"True s₀ (inference reference states, n={num_systems_inference}): E/N = {Eps_s0_true_all:.4f}, |⟨m⟩| = {M_s0_true_all:.4f}")

print(f"MLP s₀ predictions (averaged across timesteps t=1..{T-1}, n={num_systems_inference}):")
print(f"  E/N = {np.nanmean(mean_Eps_s0_pred):.4f}, |⟨m⟩| = {np.nanmean(mean_absM_s0_pred):.4f}")

# Plot energy and magnetization evolution for s0 predictions
times = np.arange(T)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# ---- Energy per spin ----
if not USE_SAVED_MODEL:
    axes[0].plot(times, mean_Eps_forward, 'o--', label='Forward', color='tab:blue', markersize=6)
axes[0].plot(times[1:], mean_Eps_reverse[1:], 'o-',  label='Reverse', color='tab:orange', markersize=6)
axes[0].plot(times[1:], mean_Eps_s0_pred[1:], 's-', label='MLP s₀ pred', color='tab:red', markersize=6, linewidth=2)
axes[0].axhline(y=Eps_s0_true_all, color='green', linestyle=':', linewidth=2, label='True s₀ (reference)', alpha=0.7)
axes[0].set_xlabel('Diffusion timestep t')
axes[0].set_ylabel('⟨E/N⟩')
axes[0].set_title('Mean energy per spin')
axes[0].grid(alpha=0.3)
axes[0].legend()

# ---- |Magnetization| ----
if not USE_SAVED_MODEL:
    axes[1].plot(times, mean_absM_forward, 'o--', label='Forward', color='tab:blue', markersize=6)
axes[1].plot(times[1:], mean_absM_reverse[1:], 'o-',  label='Reverse', color='tab:orange', markersize=6)
axes[1].plot(times[1:], mean_absM_s0_pred[1:], 's-', label='MLP s₀ pred', color='tab:red', markersize=6, linewidth=2)
axes[1].axhline(y=M_s0_true_all, color='green', linestyle=':', linewidth=2, label='True s₀ (reference)', alpha=0.7)
axes[1].set_xlabel('Diffusion timestep t')
axes[1].set_ylabel('|⟨m⟩|')
axes[1].set_title('Absolute mean magnetization')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.suptitle(f'MLP s₀ Predictions vs Forward/Reverse States (L={int(L)})\nForward: ALL training (n={num_systems}), Reverse: generated (n={num_systems_inference}), Chains={num_samples}', 
             y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig("plot_2_forwardandrev_trajs_0p453125_corr_absofmean.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
# === Terminal State Analysis: Compare initial s0 vs final reverse s0 ===
print("\n=== Terminal State Analysis ===")
print("Comparing ALL FPGA data vs generated samples")
print(f"Using device: {device}")

# Extract terminal states
# All training data: the complete set we started with
s0_all_train = states_all  # shape: (num_systems, N)
# Generated: the final states from reverse diffusion at t=0
s0_generated = s_reverse_all[:, 1, :]  # shape: (num_systems_inference, N)

# Compute magnetization per spin and energy per spin
s0_train_t = torch.from_numpy(s0_all_train.astype(np.float32)).to(device)
s0_gen_t = torch.from_numpy(s0_generated.astype(np.float32)).to(device)
J_energy = torch.from_numpy(J_original.astype(np.float32)).to(device)
h_energy = torch.from_numpy(h.astype(np.float32)).to(device)

M_all_train = _magnetization_per_spin_batch(s0_train_t).cpu().numpy()
M_generated = _magnetization_per_spin_batch(s0_gen_t).cpu().numpy()
E_all_train = _energy_per_spin_batch(s0_train_t, J_energy, h_energy).cpu().numpy()
E_generated = _energy_per_spin_batch(s0_gen_t, J_energy, h_energy).cpu().numpy()

print(f"\n[ALL FPGA DATA] Original samples (n={num_systems}):")
print(f"  Mean magnetization: {M_all_train.mean():.4f} ± {M_all_train.std():.4f}")
print(f"  Mean energy/N: {E_all_train.mean():.4f} ± {E_all_train.std():.4f}")

print(f"\n[GENERATED] Reverse diffusion samples (n={num_systems_inference}):")
print(f"  Mean magnetization: {M_generated.mean():.4f} ± {M_generated.std():.4f}")
print(f"  Mean energy/N: {E_generated.mean():.4f} ± {E_generated.std():.4f}")

In [ ]:
# --- Plot 1: Magnetization Distribution (staircase) ---
print("\nGenerating Plot 1: Magnetization Distribution...")
num_bins_m = 100
bins_m = np.linspace(-1.0, 1.0, num_bins_m)

counts_train, edges = np.histogram(M_all_train, bins=bins_m, density=False)
counts_gen, _ = np.histogram(M_generated, bins=bins_m, density=False)

counts_train = counts_train / counts_train.sum()
counts_gen = counts_gen / counts_gen.sum()

centers = 0.5 * (edges[1:] + edges[:-1])

plt.figure(figsize=(10, 6))
plt.step(centers, counts_train, where='mid', color='tab:blue', 
         label=f'All FPGA Original (n={num_systems})', linewidth=2)
plt.step(centers, counts_gen, where='mid', color='tab:orange', 
         label=f'Generated Reverse Diffusion (n={num_systems_inference})', linewidth=2, linestyle='--')
plt.xlabel('Magnetization per spin (m)', fontsize=12)
plt.ylabel('Probability', fontsize=12)
plt.title(f'Mean Magnetization Distribution (2D Ferromagnet β = {beta_start})', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 2: Absolute Magnetization Distribution (staircase) ---
print("Generating Plot 2: Absolute Magnetization Distribution...")
num_bins_absm = 100
bins_absm = np.linspace(0.0, 1.0, num_bins_absm)

absM_train = np.abs(M_all_train)
absM_gen = np.abs(M_generated)

counts_train_absm, edges_absm = np.histogram(absM_train, bins=bins_absm, density=False)
counts_gen_absm, _ = np.histogram(absM_gen, bins=bins_absm, density=False)

counts_train_absm = counts_train_absm / counts_train_absm.sum()
counts_gen_absm = counts_gen_absm / counts_gen_absm.sum()

centers_absm = 0.5 * (edges_absm[1:] + edges_absm[:-1])

plt.figure(figsize=(10, 6))
plt.step(centers_absm, counts_train_absm, where='mid', color='tab:purple', 
         label=f'All FPGA Original (n={num_systems})', linewidth=2)
plt.step(centers_absm, counts_gen_absm, where='mid', color='tab:cyan', 
         label=f'Generated Reverse Diffusion (n={num_systems_inference})', linewidth=2, linestyle='--')
plt.xlabel('Absolute magnetization per spin (|m|)', fontsize=12)
plt.ylabel('Probability', fontsize=12)
plt.title(f'Absolute Magnetization Distribution (2D Ferromagnet β = {beta_start})', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 3: Energy Distribution (staircase) ---
print("Generating Plot 3: Energy Distribution...")
num_bins_e = 50
x_min = -1.8
x_max = -0.6
bins_e = np.linspace(x_min, x_max, num_bins_e + 1)

counts_train_e, edges_e = np.histogram(
    E_all_train, bins=bins_e, density=False
)
counts_gen_e, _ = np.histogram(
    E_generated, bins=bins_e, density=False
)

counts_train_e = counts_train_e / counts_train_e.sum()
counts_gen_e = counts_gen_e / counts_gen_e.sum()

centers_e = 0.5 * (edges_e[1:] + edges_e[:-1])

plt.figure(figsize=(10, 6))
plt.step(centers_e, counts_train_e, where='mid', color='tab:green', 
         label=f'All MCMC Original (n={num_systems})', linewidth=2)
plt.step(centers_e, counts_gen_e, where='mid', color='tab:red', 
         label=f'Generated Reverse Diffusion (n={num_systems_inference})', linewidth=2, linestyle='--')
plt.xlabel('Energy per spin (E/N)', fontsize=12)
plt.ylabel('Probability', fontsize=12)
plt.title(f'Energy per Spin Distribution (3D Spin Glass β = {beta_start})', fontsize=14, fontweight='bold')
plt.xlim(x_min, x_max)

plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig("plot_2_energy_vs_beta_0p453125-corr.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
# --- Plot 4: Energy vs Magnetization Scatter ---
print("Generating Plot 4: Energy vs Magnetization Scatter...")
plt.figure(figsize=(10, 6))
plt.scatter(M_all_train, E_all_train, alpha=0.5, s=30, 
            color='tab:blue', label=f'All FPGA Original (n={num_systems})', edgecolors='none')
plt.scatter(M_generated, E_generated, alpha=0.7, s=40, 
            color='tab:orange', label=f'Generated Reverse Diffusion (n={num_systems_inference})', 
            marker='x', linewidths=1.5)
plt.xlabel('Magnetization per spin (m)', fontsize=12)
plt.ylabel('Energy per spin (E/N)', fontsize=12)
plt.title(f'Energy vs Magnetization (2D Ferromagnet β = {beta_start})', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig("plot_2_energy_mag_cloud_0p453125-corr.pdf", format="pdf", bbox_inches="tight")
plt.show()

print("\nTerminal state analysis complete.")
print(f"\nSummary: Comparing {num_systems} training samples vs {num_systems_inference} generated samples")

In [ ]:
# === Edwards-Anderson order parameter (qEA) and overlap samples ===

def compute_overlaps(states, num_pairs=10_000, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    states = np.asarray(states)
    S, N = states.shape

    # No pairs possible
    if S < 2:
        return np.empty((0,), dtype=np.float32)

    max_pairs = S*(S-1)//2
    num_pairs = min(int(num_pairs), max_pairs)

    pair_ids = rng.choice(max_pairs, size=num_pairs, replace=False).astype(np.int64)

    counts = np.arange(S-1, 0, -1, dtype=np.int64)  # S-1, S-2, ..., 1
    cum = np.cumsum(counts)

    i = np.searchsorted(cum, pair_ids, side="right").astype(np.int64)
    prev = np.concatenate(([0], cum[:-1]))
    j = (pair_ids - prev[i] + (i + 1)).astype(np.int64)

    overlaps = (states[i] * states[j]).mean(axis=1).astype(np.float32)
    return overlaps
    
def compute_EA_order_parameter(overlaps: np.ndarray) -> float:
    if overlaps.size == 0:
        return float("nan")
    return float(np.mean(overlaps**2))

print("\n=== Edwards-Anderson parameter from replica overlaps ===")

# Use the same RNG object as the rest of the notebook
num_pairs_overlap = 100_000

# Training data: original s0 samples
q_train = compute_overlaps(s0_all_train, num_pairs=num_pairs_overlap, rng=rng)
qEA_train = compute_EA_order_parameter(q_train)

# Generated data: reverse-diffusion terminal states
q_gen = compute_overlaps(s0_generated, num_pairs=num_pairs_overlap, rng=rng)
qEA_gen = compute_EA_order_parameter(q_gen)

print(f"q_EA (FPGA original s₀)   = {qEA_train:.4f}")
print(f"q_EA (generated s₀) = {qEA_gen:.4f}")

# --- Bar chart for qEA (training vs generated) ---
plt.figure(figsize=(6, 5))
labels = ['FPGA original s₀', 'Generated s₀']
q_values = [qEA_train, qEA_gen]
x = np.arange(len(labels))
barw = 0.6

bars = plt.bar(x, q_values, width=barw,
               color=['tab:blue', 'tab:orange'],
               alpha=0.85)

for xi, val in zip(x, q_values):
    plt.text(xi, val, f'{val:.4f}', ha='center', va='bottom', fontsize=11)

plt.xticks(x, labels)
plt.ylabel(r'$q_{\mathrm{EA}} = \langle q^2 \rangle$')
plt.title(fr'Edwards-Anderson Order Parameter at $\beta = {beta_start}$')
plt.ylim(0, max(q_values) * 1.15)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("plot_2_orderparam_0p453125-corr.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
# === Parisi overlap distribution P(q): training vs generated ===

print("\n=== Parisi overlap distribution P(q) ===")

num_bins_q = 101
bins_q = np.linspace(-1.0, 1.0, num_bins_q)

counts_train_q, edges_q = np.histogram(q_train, bins=bins_q, density=False)
counts_gen_q,   _       = np.histogram(q_gen,   bins=bins_q, density=False)

# Normalize to get probabilities (∑ P = 1)
counts_train_q = counts_train_q / counts_train_q.sum()
counts_gen_q   = counts_gen_q   / counts_gen_q.sum()

centers_q = 0.5 * (edges_q[1:] + edges_q[:-1])

plt.figure(figsize=(10, 6))
plt.step(centers_q, counts_train_q, where='mid',
         color='tab:blue', linewidth=2,
         label=f'FPGA original s₀ (n_pairs={len(q_train)})')
plt.step(centers_q, counts_gen_q, where='mid',
         color='tab:orange', linewidth=2, linestyle='--',
         label=f'Generated s₀ (n_pairs={len(q_gen)})')

plt.xlabel(r'Overlap $q_{\alpha\beta} = \frac{1}{N}\sum_i s_i^{(\alpha)} s_i^{(\beta)}$')
plt.ylabel(r'Probability $P(q)$')
plt.title(fr'Parisi Overlap Distribution $P(q)$ at $\beta = {beta_start}$')
plt.xlim(-1.0, 1.0)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("plot_2_overlaps_0p453125-corr.pdf", format="pdf", bbox_inches="tight")
plt.show()